In [1]:
import sys
print(sys.executable)
print(sys.version)

/common/home/projectgrps/CS425/CS425G8/jupyterlab-venv-pytorch-240/bin/python3
3.11.11 (main, Jan  2 2025, 15:36:49) [GCC 13.3.0]


In [2]:
import sys
!{sys.executable} -m pip install torch torchvision torchaudio


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [3]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130
CUDA available: False


In [4]:
import sys

!{sys.executable} -m pip install transformers datasets pandas scikit-learn sentencepiece accelerate evaluate


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [5]:
import torch
import transformers
import datasets
import pandas
import sklearn

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
gpu_available = torch.cuda.is_available()
print("GPU available:", gpu_available)

if gpu_available:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU for inference and testing.")

/common/home/projectgrps/CS425/CS425G8/jupyterlab-venv-pytorch-240/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.14.1+cu130
Transformers: 5.20.1
Datasets: 5.1.0
GPU available: False
Using CPU for inference and testing.


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

model = model.to(device)
model.eval()

prompt = "Normalize this Singlish sentence into standard English: Can lah, I go there now."

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=128
).to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=60,
        num_beams=4
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))
print("Model device:", next(model.parameters()).device)


Using device: cpu


Loading weights: 100%|████████████████████████████████████████████████████████████| 282/282 [00:00<00:00, 11348.46it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


I'll go there now.
Model device: cpu


In [7]:
from datasets import load_dataset

dataset = load_dataset(
    "kumarsujit474/singlish-english-aligned-translation-seat"
)

print(dataset)
print(dataset["train"].column_names)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['singlish', 'english'],
        num_rows: 1603
    })
})
['singlish', 'english']
{'singlish': 'having prac at 12pm sia', 'english': 'having practise at 12pm goodness'}


In [8]:
import os
import pandas as pd

os.makedirs("cs425_data", exist_ok=True)

df = dataset["train"].to_pandas()
df.to_csv("cs425_data/singlish_english.csv", index=False)

print("Dataset shape:", df.shape)
print(df.head(5))

Dataset shape: (1603, 2)
                                            singlish  \
0                            having prac at 12pm sia   
1  My brother, he write until very late every nig...   
2                                Change for you liao   
3                                     Booked uber le   
4                              don't like chilli sia   

                                             english  
0                   having practise at 12pm goodness  
1  My brother writes until very late every night;...  
2                             change for you already  
3                                 book ubver already  
4                                  don't like chilli  


In [9]:
import pandas as pd

df = pd.read_csv("cs425_data/singlish_english.csv")

print("Dataset size:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

print("\nExact duplicate pairs:", df.duplicated().sum())
print("\nDuplicate Singlish inputs:", df["singlish"].duplicated().sum())

# Inspect random examples
display(df.sample(min(15, len(df)), random_state=42))

Dataset size: (1603, 2)

Missing values:
singlish    0
english     0
dtype: int64

Exact duplicate pairs: 2

Duplicate Singlish inputs: 3


,singlish,english
528,No lah just kidding,"No, just kidding"
1145,Lunch jio at 10am,lunch invite at 10am
168,All not on my list La,all not on my list
135,I morning got photoshoot leh no go,I morning got photoshoot though no go
1330,we Find one day makan tgt aFter intern okk,we find one day eat together after intern ok
940,"I yesterday already reply to your email, you d...","I replied to your email yesterday, didn't you ..."
1132,Have is have la but sometimes a bit paiseh to ...,Have is have but sometimes a bit embarrassed t...
237,Simi oh ya,what oh yea
481,last call alr lai lai,last call already come come
1034,"Eh, you know the bus fare how much ah?","Hey, do you know how much the bus fare is?"


In [10]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 1282
Validation: 160
Test: 161


In [11]:
train_df.to_csv("cs425_data/train.csv", index=False)
val_df.to_csv("cs425_data/validation.csv", index=False)
test_df.to_csv("cs425_data/test.csv", index=False)

print("All three splits saved.")

All three splits saved.


In [17]:

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "/common/home/projectgrps/CS425/CS425G8/cs425_data/flan_t5_cleaned/final_model"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
model.eval()

test_df = pd.read_csv("cs425_data/test.csv")

def normalize_singlish(text):
    prompt = (
        "Normalize this Singlish sentence into natural standard English. "
        "Preserve its meaning.\n"
        f"Singlish: {text}\n"
        "English:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            num_beams=4
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test on the first 10 examples
sample = test_df.head(10).copy()
sample["prediction"] = sample["singlish"].apply(normalize_singlish)

display(sample[["singlish", "english", "prediction"]])

Using device: cpu


Loading weights: 100%|██████████████████████████████████████████████████████████████| 282/282 [00:01<00:00, 279.19it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


,singlish,english,prediction
0,Can buy anything you want la,Can buy anything you want,Can buy anything you want
1,Best don't kena accident lah,best don't get into accident,Best don't have an accident
2,My teacher say must read more books to improve...,My teacher said that we must read more books t...,My teacher said we should read more books to i...
3,"Eh, you know the bus fare how much ah?","Hey, do you know how much the bus fare is?","Hey, do you know the bus fare?"
4,"Wah, your shading technique quite solid hor.","Wow, your shading technique is quite impressive.","Wow, your shading technique is quite solid."
5,"Eh, you already shower or not?","Hey, have you taken a shower yet?","Hey, have you already taken a shower?"
6,"I saw your text, but I was in a meeting just n...","I saw your text, but I was in a meeting and co...","I saw your text, but I was in a meeting just n..."
7,At first the queue damn long sia,At first the queue damn long already,At first the queue was very long
8,Love you guys lah,love you guys,Love you guys
9,So sit lepak 1 ah haha,So sit and relax 1 ah haha,So let's sit for 1 haha


In [18]:

from IPython.display import display
import pandas as pd
import time

results = []
start_time = time.time()

for i, row in test_df.iterrows():
    prediction = normalize_singlish(row["singlish"])

    results.append({
        "singlish": row["singlish"],
        "english": row["english"],
        "prediction": prediction
    })

    if (len(results)) % 10 == 0:
        print(f"Processed {len(results)}/{len(test_df)} examples")

results_df = pd.DataFrame(results)

print(f"\nTotal examples: {len(results_df)}")
print(f"Time taken: {(time.time() - start_time) / 60:.1f} minutes")

display(results_df.head(20))


Processed 10/161 examples
Processed 20/161 examples
Processed 30/161 examples
Processed 40/161 examples
Processed 50/161 examples
Processed 60/161 examples
Processed 70/161 examples
Processed 80/161 examples
Processed 90/161 examples
Processed 100/161 examples
Processed 110/161 examples
Processed 120/161 examples
Processed 130/161 examples
Processed 140/161 examples
Processed 150/161 examples
Processed 160/161 examples

Total examples: 161
Time taken: 10.3 minutes


,singlish,english,prediction
0,Can buy anything you want la,Can buy anything you want,Can buy anything you want
1,Best don't kena accident lah,best don't get into accident,Best don't have an accident
2,My teacher say must read more books to improve...,My teacher said that we must read more books t...,My teacher said we should read more books to i...
3,"Eh, you know the bus fare how much ah?","Hey, do you know how much the bus fare is?","Hey, do you know the bus fare?"
4,"Wah, your shading technique quite solid hor.","Wow, your shading technique is quite impressive.","Wow, your shading technique is quite solid."
5,"Eh, you already shower or not?","Hey, have you taken a shower yet?","Hey, have you already taken a shower?"
6,"I saw your text, but I was in a meeting just n...","I saw your text, but I was in a meeting and co...","I saw your text, but I was in a meeting just n..."
7,At first the queue damn long sia,At first the queue damn long already,At first the queue was very long
8,Love you guys lah,love you guys,Love you guys
9,So sit lepak 1 ah haha,So sit and relax 1 ah haha,So let's sit for 1 haha


In [20]:

# Evaluate the existing predictions without rerunning the model

print("Total test examples:", len(results_df))

# Case-insensitive exact match
exact_match = (
    results_df["prediction"].str.strip().str.casefold()
    == results_df["english"].str.strip().str.casefold()
).mean()

print(f"Exact match (case-insensitive): {exact_match:.1%}")

# Show predictions that differ from the reference translation
different = results_df[
    results_df["prediction"].str.strip().str.casefold()
    != results_df["english"].str.strip().str.casefold()
].copy()

print(f"Non-exact matches: {len(different)} / {len(results_df)}")

display(different.head(25))


Total test examples: 161
Exact match (case-insensitive): 21.1%
Non-exact matches: 127 / 161


,singlish,english,prediction
1,Best don't kena accident lah,best don't get into accident,Best don't have an accident
2,My teacher say must read more books to improve...,My teacher said that we must read more books t...,My teacher said we should read more books to i...
3,"Eh, you know the bus fare how much ah?","Hey, do you know how much the bus fare is?","Hey, do you know the bus fare?"
4,"Wah, your shading technique quite solid hor.","Wow, your shading technique is quite impressive.","Wow, your shading technique is quite solid."
5,"Eh, you already shower or not?","Hey, have you taken a shower yet?","Hey, have you already taken a shower?"
6,"I saw your text, but I was in a meeting just n...","I saw your text, but I was in a meeting and co...","I saw your text, but I was in a meeting just n..."
7,At first the queue damn long sia,At first the queue damn long already,At first the queue was very long
9,So sit lepak 1 ah haha,So sit and relax 1 ah haha,So let's sit for 1 haha
10,"I so sleepy already, need to power nap a bit.","I'm very sleepy, I need to take a power nap.","I'm so sleepy, I need to power nap a bit."
11,"Wah, this author's style of writing very chim,...","Wow, this author's writing style is very profo...","Wow, this author's style of writing is very in..."


In [21]:

import re

def has_negation(text):
    return bool(re.search(
        r"\b(no|not|never|don't|doesn't|didn't|can't|cannot|without|avoid)\b",
        str(text).lower()
    ))

def protected_terms(text):
    terms = [
        "chicken rice", "nasi lemak", "nasi padang",
        "mee goreng", "bubble tea", "prata", "laksa",
        "mala", "satay", "seafood", "peanuts",
        "vegetarian", "halal", "no pork", "no beef",
        "chicken", "beef", "pork"
    ]
    text = str(text).lower()
    return [term for term in terms if re.search(
        r"\b" + re.escape(term) + r"\b", text
    )]

issues = []

for _, row in results_df.iterrows():
    original = str(row["singlish"])
    prediction = str(row["prediction"])

    reasons = []

    if has_negation(original) and not has_negation(prediction):
        reasons.append("Possible lost negation")

    missing = [
        term for term in protected_terms(original)
        if term not in prediction.lower()
    ]
    if missing:
        reasons.append("Possible lost food/constraint term: " + ", ".join(missing))

    if original.rstrip().endswith("?") != prediction.rstrip().endswith("?"):
        reasons.append("Question punctuation changed")

    if reasons:
        issues.append({
            "singlish": original,
            "prediction": prediction,
            "potential_issue": "; ".join(reasons)
        })

issues_df = pd.DataFrame(issues)

print("Potential safety issues:", len(issues_df), "out of", len(results_df))
display(issues_df.head(30))


Potential safety issues: 13 out of 161


,singlish,prediction,potential_issue
0,"Eh, you already shower or not?","Hey, have you already taken a shower?",Possible lost negation
1,"I shy lah, where got people want to see me dance.","I'm shy, where do people want to see me dance?",Question punctuation changed
2,Already beautiful lah what you talking about,"Already beautiful, what are you talking about?",Question punctuation changed
3,HAVE YALL DIED ALR,Have you guys gone insane?,Question punctuation changed
4,"Eh, you know where got good sewing machine not?","Hey, do you know where to get a good sewing ma...",Possible lost negation
5,You know is which one meh,You know is which one?,Question punctuation changed
6,what difficulty sia,What difficulty is it?,Question punctuation changed
7,"Eh, you know how to sew button or not?","Hey, do you know how to sew a button?",Possible lost negation
8,Can lah in the night,Can you sleep in the night?,Question punctuation changed
9,You all want to come my house play together or...,Do you all want to come to my house to play to...,Possible lost negation


In [22]:

import re
import pandas as pd

NEGATION_PATTERNS = [
    r"\bno\b",
    r"\bnot\b",
    r"\bnever\b",
    r"\bdon't\b",
    r"\bdoesn't\b",
    r"\bdidn't\b",
    r"\bcan't\b",
    r"\bcannot\b",
    r"\bwithout\b",
    r"\bavoid\b",
    r"\bexcept\b",
]

PROTECTED_TERMS = [
    "chicken rice", "nasi lemak", "nasi padang", "mee goreng",
    "bubble tea", "prata", "laksa", "mala", "satay",
    "seafood", "peanuts", "vegetarian", "halal",
    "chicken", "beef", "pork"
]

def contains_pattern(text, patterns):
    text = str(text).casefold()
    return any(re.search(pattern, text) for pattern in patterns)

def find_protected_terms(text):
    text = str(text).casefold()
    return [
        term for term in PROTECTED_TERMS
        if re.search(r"\b" + re.escape(term) + r"\b", text)
    ]

def inspect_safety(original, prediction):
    reasons = []

    # Exclude Singlish yes/no questions ending in "or not"
    original_for_negation = re.sub(
        r"\bor not\s*\??\s*$", "", str(original), flags=re.IGNORECASE
    )

    if (
        contains_pattern(original_for_negation, NEGATION_PATTERNS)
        and not contains_pattern(prediction, NEGATION_PATTERNS)
    ):
        reasons.append("Check whether a negative constraint was lost")

    missing_terms = [
        term for term in find_protected_terms(original)
        if term not in str(prediction).casefold()
    ]

    if missing_terms:
        reasons.append("Check missing terms: " + ", ".join(missing_terms))

    return reasons

review_rows = []

for _, row in results_df.iterrows():
    reasons = inspect_safety(row["singlish"], row["prediction"])
    if reasons:
        review_rows.append({
            "singlish": row["singlish"],
            "prediction": row["prediction"],
            "potential_issue": "; ".join(reasons)
        })

review_df = pd.DataFrame(review_rows)

print("Examples needing manual review:", len(review_df), "out of", len(results_df))
display(review_df)


Examples needing manual review: 3 out of 161


,singlish,prediction,potential_issue
0,"Eh, you know where got good sewing machine not?","Hey, do you know where to get a good sewing ma...",Check whether a negative constraint was lost
1,"Eh, you know not? Singapore got a lot of ERP g...","Hey, do you know? Singapore has a lot of ERP g...",Check whether a negative constraint was lost
2,You watch the news not? Got a lot of important...,Have you watched the news? There are a lot of ...,Check whether a negative constraint was lost


In [17]:

# Test the fine-tuned checkpoint with the training prompt

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

checkpoint = "/common/home/projectgrps/CS425/CS425G8/cs425_data/flan_t5_cleaned/final_model"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

test_tokenizer = AutoTokenizer.from_pretrained(checkpoint)
test_model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint).to(device)
test_model.eval()

def flan_normalize(text):
    prompt = f"Normalize this Singlish into natural English: {text}"
    inputs = test_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = test_model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=4
        )

    return test_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

test_queries = [
    "I want chicken rice, no chilli please.",
    "I don't eat seafood.",
    "Got vegetarian options or not?",
    "Can lah, I go there now.",
    "Those cafes quite ulu I think."
]

for query in test_queries:
    print("Input: ", query)
    print("Output:", flan_normalize(query))
    print("-" * 60)


Using device: cpu


Loading weights: 100%|██████████████████████████████████████████████████████████████| 282/282 [00:00<00:00, 310.99it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Input:  I want chicken rice, no chilli please.
Output: I want chicken rice, no chilli please.
------------------------------------------------------------
Input:  I don't eat seafood.
Output: I don't eat seafood.
------------------------------------------------------------
Input:  Got vegetarian options or not?
Output: Have you got vegetarian options?
------------------------------------------------------------
Input:  Can lah, I go there now.
Output: Can I go there now?
------------------------------------------------------------
Input:  Those cafes quite ulu I think.
Output: Those cafes are quite expensive, I think.
------------------------------------------------------------


In [18]:

print("SINGLISH_RULES defined:", "SINGLISH_RULES" in globals())
print("PROTECTED_TERMS defined:", "PROTECTED_TERMS" in globals())
print("preprocess_singlish defined:", "preprocess_singlish" in globals())
print("integrated_normalize defined:", "integrated_normalize" in globals())
print("safe_integrated_normalize defined:", "safe_integrated_normalize" in globals())

if "SINGLISH_RULES" in globals():
    print("\nExisting Singlish rules:")
    print(SINGLISH_RULES)


SINGLISH_RULES defined: False
PROTECTED_TERMS defined: False
preprocess_singlish defined: True
integrated_normalize defined: True
safe_integrated_normalize defined: True


In [19]:
import sys
!{sys.executable} -m pip install sacrebleu


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: /common/home/projectgrps/CS425/CS425G8/jupyterlab-venv-pytorch-240/bin/python3 -m pip install --upgrade pip


In [20]:
import transformers
import inspect
from transformers import Seq2SeqTrainingArguments

print("Transformers:", transformers.__version__)
print(inspect.signature(Seq2SeqTrainingArguments))

Transformers: 5.20.1
(output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 5e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool = False, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = False, gradient_checkpointing_kwargs: dict[str, typing.Any] | str | None = None, torch_compile: bool = 

In [21]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
)

MODEL_NAME = "google/flan-t5-base"
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128

train_df = pd.read_csv("cs425_data/train.csv")
val_df = pd.read_csv("cs425_data/validation.csv")
test_df = pd.read_csv("cs425_data/test.csv")

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("GPU:", torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model = model.to("cuda")

def preprocess(batch):
    inputs = [
        "Normalize this Singlish sentence into natural standard English. "
        "Preserve its meaning.\nSinglish: " + text
        for text in batch["singlish"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    labels = tokenizer(
        text_target=batch["english"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds = Dataset.from_pandas(val_df, preserve_index=False)

train_tokenized = train_ds.map(
    preprocess, batched=True, remove_columns=train_ds.column_names
)
val_tokenized = val_ds.map(
    preprocess, batched=True, remove_columns=val_ds.column_names
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

print("Data preprocessing complete.")

Train: 1282
Validation: 160
Test: 161


RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="cs425_data/flan_t5_checkpoints",
    num_train_epochs=3,
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=25,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=42,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("Trainer configured.")

In [ ]:
train_result = trainer.train()

print("Training complete.")
print(train_result.metrics)

In [ ]:
FINAL_MODEL_DIR = "cs425_data/flan_t5_singlish_final"

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("Model saved to:", FINAL_MODEL_DIR)

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)

MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model = model.to("cuda")

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

print("Fresh pretrained model loaded.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="cs425_data/flan_t5_fp32_check",
    num_train_epochs=3,
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,
    fp16=False,
    bf16=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=25,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=42,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("Fresh trainer configured without mixed precision.")

In [ ]:
train_result = trainer.train()

print("Training complete.")
print(train_result.metrics)

In [ ]:
import os

FINETUNED_PATH = "cs425_data/flan_t5_fp32_check/final_model"

# Save the trained model (best checkpoint selected by the trainer)
trainer.save_model(FINETUNED_PATH)
tokenizer.save_pretrained(FINETUNED_PATH)

# Verify the files exist
print(os.listdir(FINETUNED_PATH))

In [ ]:
finetuned_model = AutoModelForSeq2SeqLM.from_pretrained(
    FINETUNED_PATH
).to("cuda")

finetuned_model.eval()

print("Fine-tuned model loaded successfully!")

In [ ]:
import os

print("Output directory contents:")
print(os.listdir("cs425_data/flan_t5_fp32_check"))

print("\nCheckpoint directories:")
for item in os.listdir("cs425_data/flan_t5_fp32_check"):
    path = os.path.join("cs425_data/flan_t5_fp32_check", item)
    if os.path.isdir(path):
        print(item, "->", os.listdir(path)[:10])

In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-base"
FINETUNED_PATH = "cs425_data/flan_t5_fp32_check/final_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
).to("cuda")

finetuned_model = AutoModelForSeq2SeqLM.from_pretrained(
    FINETUNED_PATH
).to("cuda")

base_model.eval()
finetuned_model.eval()

print("Both models loaded successfully!")


In [ ]:

def normalize(text, model, max_new_tokens=64):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4
        )

    return tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

examples = [
    "Can help me check this one or not?",
    "Wah, this food damn nice leh.",
    "I want makan something nearby.",
    "Can recommend one good place to eat ah?"
]

for text in examples:
    print(f"\nInput: {text}")
    print(f"Base:      {normalize(text, base_model)}")
    print(f"Fine-tuned: {normalize(text, finetuned_model)}")


In [ ]:

import os

print("Current working directory:", os.getcwd())
print("\nCSV files in cs425_data:")

if os.path.exists("cs425_data"):
    for root, dirs, files in os.walk("cs425_data"):
        for file in files:
            if file.endswith(".csv"):
                print(os.path.join(root, file))
else:
    print("cs425_data folder not found in current directory.")


In [ ]:


import pandas as pd

train_df = pd.read_csv("cs425_data/train.csv")
val_df = pd.read_csv("cs425_data/validation.csv")
test_df = pd.read_csv("cs425_data/test.csv")

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nTraining examples:")
display(train_df.sample(min(15, len(train_df)), random_state=42))


In [ ]:

patterns = r"makan|recommend|or not|leh|lah|ah"

matches = train_df[
    train_df["singlish"].str.contains(
        patterns, case=False, na=False, regex=True
    )
]

print("Matching examples:", len(matches))
display(matches.head(30))


In [ ]:

import pandas as pd

for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n--- {name} ---")
    print("Rows:", len(df))
    print("Missing Singlish:", df["singlish"].isna().sum())
    print("Missing English:", df["english"].isna().sum())
    print("Duplicate Singlish:", df["singlish"].duplicated().sum())
    print(
        "Identical input/output:",
        (
            df["singlish"].str.strip().str.lower()
            == df["english"].str.strip().str.lower()
        ).sum()
    )

print("\nPotentially problematic training pairs:")
display(
    matches[
        matches["english"].str.contains(
            r"\b(i poor|i did go it|you don't drunk|anough)\b",
            case=False,
            na=False
        )
    ]
)


In [ ]:

import pandas as pd
import torch
from tqdm.auto import tqdm

results = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    source = str(row["singlish"])
    reference = str(row["english"])

    base_prediction = normalize(source, base_model)
    finetuned_prediction = normalize(source, finetuned_model)

    results.append({
        "singlish": source,
        "reference_english": reference,
        "base_prediction": base_prediction,
        "finetuned_prediction": finetuned_prediction
    })

results_df = pd.DataFrame(results)
results_df.to_csv("cs425_data/model_comparison.csv", index=False)

print("Evaluation examples:", len(results_df))
print("Saved to cs425_data/model_comparison.csv")

display(results_df.head(20))


In [ ]:

# Sample 50 training pairs for a manual quality audit
audit_sample = train_df.sample(n=50, random_state=2026).copy()

audit_sample = audit_sample.reset_index(drop=True)
audit_sample.insert(0, "id", range(1, len(audit_sample) + 1))

# Add blank columns so we can record our review
audit_sample["translation_ok"] = ""
audit_sample["issue"] = ""

display(audit_sample[["id", "singlish", "english"]])


In [ ]:

audit_path = "cs425_data/seat_quality_audit.csv"

audit_sample.to_csv(audit_path, index=False)

print("Audit saved to:", audit_path)
print("Rows to review:", len(audit_sample))


In [ ]:

# First-pass labels based on the 50 examples you shared.
# Review and change these labels if you disagree.

labels = {
    1: ("yes", ""),
    2: ("uncertain", "Context needed to verify intended meaning"),
    3: ("uncertain", "Example is truncated"),
    4: ("yes", ""),
    5: ("yes", ""),
    6: ("yes", ""),
    7: ("no", "English remains ungrammatical"),
    8: ("uncertain", "Context needed for 'outside Cheers'"),
    9: ("yes", ""),
    10: ("uncertain", "Meaning mostly preserved; phrasing needs review"),
    11: ("uncertain", "Target is truncated"),
    12: ("yes", ""),
    13: ("uncertain", "Example is truncated"),
    14: ("uncertain", "Example is truncated"),
    15: ("uncertain", "Meaning depends on context"),
    16: ("uncertain", "Translation may not fully preserve meaning"),
    17: ("uncertain", "Meaning depends on context"),
    18: ("uncertain", "Input and target are incomplete"),
    19: ("yes", "Short phrase; acceptable as a translation"),
    20: ("yes", ""),
    21: ("yes", ""),
    22: ("uncertain", "Meaning depends on context"),
    23: ("yes", ""),
    24: ("no", "Awkward phrasing and possible meaning loss"),
    25: ("no", "Likely mistranslation: pants versus parents"),
    26: ("no", "English remains ungrammatical"),
    27: ("no", "Singlish idiom remains untranslated"),
    28: ("no", "English remains ungrammatical"),
    29: ("no", "English remains ungrammatical"),
    30: ("yes", ""),
    31: ("uncertain", "Target is truncated"),
    32: ("yes", "Reasonable slang normalization"),
    33: ("no", "Subject-verb agreement error"),
    34: ("yes", ""),
    35: ("uncertain", "Target is truncated"),
    36: ("no", "English remains ungrammatical"),
    37: ("yes", ""),
    38: ("yes", ""),
    39: ("uncertain", "Target is truncated"),
    40: ("no", "ORD may have been mistranslated as operation"),
    41: ("yes", ""),
    42: ("yes", "Meaning broadly preserved"),
    43: ("yes", ""),
    44: ("yes", ""),
    45: ("uncertain", "Target is truncated"),
    46: ("yes", ""),
    47: ("no", "Meaning of 'don't like that' is ambiguous"),
    48: ("uncertain", "Informal expression needs context"),
    49: ("yes", ""),
    50: ("no", "Likely meaning loss; needs source context"),
}

audit_sample["translation_ok"] = audit_sample["id"].map(
    lambda i: labels[i][0]
)
audit_sample["issue"] = audit_sample["id"].map(
    lambda i: labels[i][1]
)

audit_sample.to_csv(
    "cs425_data/seat_quality_audit.csv",
    index=False
)

print("First-pass audit counts:")
print(audit_sample["translation_ok"].value_counts())

print("\nExamples flagged for review:")
display(
    audit_sample[audit_sample["translation_ok"] != "yes"][
        ["id", "singlish", "english", "translation_ok", "issue"]
    ]
)


In [ ]:

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 60)

display(
    audit_sample.loc[
        audit_sample["translation_ok"].isin(["no", "uncertain"]),
        ["id", "singlish", "english", "translation_ok", "issue"]
    ].sort_values("id")
)


In [ ]:
import pandas as pd

# Start with the original training data.
clean_train = train_df.copy()

corrections = {
    "I thought we gg hawker tabao":
        "I thought we were going to get takeaway from the hawker centre.",

    "then 10.45 can liao lol":
        "Then 10:45 works, lol.",

    "U dont play play hor":
        "Don't play around, okay?",

    "What she need arh":
        "What does she need?",

    "i think most ppl here alr knows who their gifter is la":
        "I think most people here already know who their gifter is.",

    "And I on vacation lah":
        "And I'm on vacation.",

    "Dun liddat leh":
        "Don't be like that.",
}

# Remove examples whose intended meaning is too uncertain to repair safely.
exclude_examples = {
    "mai lai ah hahah dont sue leh my pants will drop",
    "the other day one cmi macam practice session",
}

# Apply changes only to the training split.
clean_train = clean_train[
    ~clean_train["singlish"].isin(exclude_examples)
].copy()

for source, target in corrections.items():
    mask = clean_train["singlish"].eq(source)
    clean_train.loc[mask, "english"] = target

print("Original training rows:", len(train_df))
print("Cleaned training rows:", len(clean_train))
print("Rows removed:", len(train_df) - len(clean_train))

clean_train.to_csv(
    "cs425_data/train_cleaned_pilot.csv",
    index=False
)

In [ ]:
display(
    clean_train[
        clean_train["singlish"].isin([
            "I thought we gg hawker tabao",
            "then 10.45 can liao lol",
            "U dont play play hor",
            "What she need arh",
            "i think most ppl here alr knows who their gifter is la",
            "And I on vacation lah",
            "Dun liddat leh",
        ])
    ][["singlish", "english"]]
)

In [ ]:
def preprocess_function(examples):
    inputs = [
        "Normalize this Singlish into natural English: " + str(text)
        for text in examples["singlish"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=128,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["english"],
        max_length=128,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
from datasets import Dataset

# Convert pandas DataFrame to Hugging Face Dataset
clean_train_dataset = Dataset.from_pandas(
    clean_train.reset_index(drop=True)
)

# Tokenize in batches
train_tokenized_clean = clean_train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=clean_train_dataset.column_names
)

print(train_tokenized_clean)

In [ ]:
print(train_tokenized_clean[0])

In [ ]:
from datasets import Dataset

validation_df = pd.read_csv("cs425_data/validation.csv")

validation_dataset = Dataset.from_pandas(
    validation_df.reset_index(drop=True)
)

val_tokenized_clean = validation_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=validation_dataset.column_names
)

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

MODEL_NAME = "google/flan-t5-base"

# Start from the original pretrained model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

training_args = Seq2SeqTrainingArguments(
    output_dir="cs425_data/flan_t5_cleaned",
    num_train_epochs=3,
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    fp16=False,
    bf16=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=25,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=42,
)

trainer_clean = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized_clean,
    eval_dataset=val_tokenized_clean,
    data_collator=data_collator,
    processing_class=tokenizer
)

train_result_clean = trainer_clean.train()

In [ ]:
final_path = "cs425_data/flan_t5_cleaned/final_model"

trainer_clean.save_model(final_path)
tokenizer.save_pretrained(final_path)

print("Saved cleaned model to:", final_path)

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import Dataset

MODEL_NAME = "google/flan-t5-base"

test_df = pd.read_csv("cs425_data/test.csv")
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

test_tokenized = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names
)

# Load original fine-tuned model
original_model = AutoModelForSeq2SeqLM.from_pretrained(
    "cs425_data/flan_t5_fp32_check/final_model"
).to("cuda")

# Load cleaned fine-tuned model
cleaned_model = AutoModelForSeq2SeqLM.from_pretrained(
    "cs425_data/flan_t5_cleaned/final_model"
).to("cuda")

original_model.eval()
cleaned_model.eval()

print("Test examples:", len(test_df))

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=cleaned_model,
    padding=True
)

test_loader = DataLoader(
    test_tokenized,
    batch_size=4,
    collate_fn=data_collator
)

def calculate_test_loss(model, test_loader):
    model.eval()
    losses = []
    total_examples = 0

    with torch.no_grad():
        for batch in test_loader:
            batch = {
                key: value.to("cuda")
                for key, value in batch.items()
            }

            outputs = model(**batch)

            batch_size = batch["input_ids"].size(0)
            losses.append(outputs.loss.item() * batch_size)
            total_examples += batch_size

    return sum(losses) / total_examples

original_test_loss = calculate_test_loss(
    original_model, test_loader
)
cleaned_test_loss = calculate_test_loss(
    cleaned_model, test_loader
)

print(f"Original model test loss: {original_test_loss:.4f}")
print(f"Cleaned model test loss:  {cleaned_test_loss:.4f}")

In [ ]:
def generate_translation(text, model):
    inputs = tokenizer(
        "Normalize this Singlish into natural English: " + str(text),
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=4
        )

    return tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

# Use the same randomly selected test examples for both models
sample = test_df.sample(n=20, random_state=42).copy()

sample["original_prediction"] = sample["singlish"].apply(
    lambda x: generate_translation(x, original_model)
)

sample["cleaned_prediction"] = sample["singlish"].apply(
    lambda x: generate_translation(x, cleaned_model)
)

display(
    sample[
        ["singlish", "english",
         "original_prediction", "cleaned_prediction"]
    ].reset_index(drop=True)
)

In [ ]:

import pandas as pd
import torch
from tqdm.auto import tqdm

device = "cuda"

def generate_predictions(texts, model, batch_size=8):
    model.eval()
    predictions = []

    for start in tqdm(range(0, len(texts), batch_size)):
        batch_texts = [
            "Normalize this Singlish into natural English: " + str(text)
            for text in texts[start:start + batch_size]
        ]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        ).to(device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=64,
                num_beams=4
            )

        predictions.extend(
            tokenizer.batch_decode(
                output_ids,
                skip_special_tokens=True
            )
        )

    return predictions


# Generate predictions for the full test set
test_results = test_df.copy()

test_results["original_prediction"] = generate_predictions(
    test_results["singlish"].tolist(),
    original_model
)

test_results["cleaned_prediction"] = generate_predictions(
    test_results["singlish"].tolist(),
    cleaned_model
)

# Save the comparison
output_path = "cs425_data/test_predictions_comparison.csv"
test_results.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Number of test examples:", len(test_results))


In [ ]:
display(
    test_results[
        [
            "singlish",
            "english",
            "original_prediction",
            "cleaned_prediction"
        ]
    ].sample(n=30, random_state=42).reset_index(drop=True)
)

In [ ]:
restaurant_tests = [
    "Got any good makan place nearby?",
    "Don't want too ex lah.",
    "Got vegetarian options anot?",
    "I feel like eating chicken rice leh.",
    "Can find somewhere not so crowded one?",
    "I want something siew dai.",
    "Any place got cheap and good food?",
    "The cafe quite ulu, got anything nearer?",
    "Can recommend one place with no seafood?",
    "We want makan, but don't want to queue so long."
]

restaurant_test_results = pd.DataFrame({
    "singlish": restaurant_tests
})

restaurant_test_results["original_prediction"] = generate_predictions(
    restaurant_tests, original_model
)

restaurant_test_results["cleaned_prediction"] = generate_predictions(
    restaurant_tests, cleaned_model
)

display(restaurant_test_results)

In [ ]:
import pandas as pd

restaurant_eval = [
    # Budget
    ("Don't want too ex lah.",
     "I don't want anything too expensive.", "budget"),
    ("Got anything cheap and good anot?",
     "Is there anything affordable and good?", "budget"),
    ("This one damn ex sia.",
     "This one is very expensive.", "budget"),
    ("Can find something within $15?",
     "Can you find something within $15?", "budget"),
    ("Not too expensive can already.",
     "Something that isn't too expensive is fine.", "budget"),

    # Food and cuisine
    ("Got any good makan place nearby?",
     "Are there any good places to eat nearby?", "food"),
    ("I feel like eating chicken rice leh.",
     "I feel like eating chicken rice.", "food"),
    ("Anything nice for supper?",
     "Is there anything good to eat for supper?", "food"),
    ("We want makan, but don't want to queue so long.",
     "We want to eat, but we don't want to queue for too long.", "food"),
    ("Can tabao this one?",
     "Can I get this takeaway?", "food"),

    # Dietary restrictions
    ("Can recommend one place with no seafood?",
     "Can you recommend a place that doesn't serve seafood?", "dietary"),
    ("Got vegetarian options anot?",
     "Are there any vegetarian options?", "dietary"),
    ("I don't eat seafood leh.",
     "I don't eat seafood.", "dietary"),
    ("Got anything without meat?",
     "Is there anything without meat?", "dietary"),
    ("My friend vegetarian, where can eat?",
     "My friend is vegetarian. Where can we eat?", "dietary"),

    # Location and distance
    ("The cafe quite ulu, got anything nearer?",
     "The cafe is quite far out. Is there anything closer?", "location"),
    ("Got any makan place near SMU?",
     "Are there any places to eat near SMU?", "location"),
    ("A bit far leh, got somewhere closer?",
     "It's a bit far. Is there somewhere closer?", "location"),
    ("Don't want to travel so far.",
     "I don't want to travel too far.", "location"),
    ("Any place walking distance?",
     "Is there anywhere within walking distance?", "location"),

    # Preferences and constraints
    ("I want something siew dai.",
     "I want something with less sugar.", "preference"),
    ("Can find somewhere not so crowded one?",
     "Can you find somewhere that isn't too crowded?", "crowds"),
    ("Don't want to wait so long leh.",
     "I don't want to wait for too long.", "queue"),
    ("Can jio my friends here?",
     "Can I invite my friends here?", "social"),
    ("This place quite shiok sia.",
     "This place is really good.", "opinion"),
]

restaurant_eval_df = pd.DataFrame(
    restaurant_eval,
    columns=["singlish", "reference", "category"]
)

print("Evaluation examples:", len(restaurant_eval_df))
display(restaurant_eval_df)

In [ ]:
restaurant_eval_df["original_prediction"] = generate_predictions(
    restaurant_eval_df["singlish"].tolist(),
    original_model
)

restaurant_eval_df["cleaned_prediction"] = generate_predictions(
    restaurant_eval_df["singlish"].tolist(),
    cleaned_model
)

display(
    restaurant_eval_df[
        ["category", "singlish", "reference",
         "original_prediction", "cleaned_prediction"]
    ]
)

restaurant_eval_df.to_csv(
    "cs425_data/restaurant_eval_comparison.csv",
    index=False
)

print("Saved to cs425_data/restaurant_eval_comparison.csv")

In [ ]:
import re

singlish_lexicon = [
    (r"\bsiew\s+dai\b", "with less sugar"),
    (r"\bdamn\s+ex\b", "very expensive"),
    (r"\btoo\s+ex\b", "too expensive"),
    (r"\bex\b", "expensive"),
    (r"\bmakan\s+place\b", "place to eat"),
    (r"\bmakan\b", "eat"),
    (r"\btabao\b", "get takeaway"),
    (r"\bulu\b", "far out"),
    (r"\bjio\b", "invite"),
    (r"\banot\b", "or not"),
]

def preprocess_singlish(text):
    text = str(text)

    for pattern, replacement in singlish_lexicon:
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)

    # Remove common conversational particles for this experiment
    text = re.sub(r"\s+\b(lah|leh|lor|sia|mah|hor)\b", "", text, flags=re.IGNORECASE)

    return re.sub(r"\s+", " ", text).strip()

restaurant_eval_df["lexicon_input"] = (
    restaurant_eval_df["singlish"].apply(preprocess_singlish)
)

display(
    restaurant_eval_df[
        ["singlish", "lexicon_input", "reference"]
    ]
)

In [ ]:
restaurant_eval_df["lexicon_prediction"] = generate_predictions(
    restaurant_eval_df["lexicon_input"].tolist(),
    cleaned_model
)

display(
    restaurant_eval_df[
        [
            "category",
            "singlish",
            "reference",
            "cleaned_prediction",
            "lexicon_input",
            "lexicon_prediction",
        ]
    ]
)

restaurant_eval_df.to_csv(
    "cs425_data/restaurant_lexicon_comparison.csv",
    index=False
)

In [ ]:
restaurant_eval_df["cleaned_correct"] = ""
restaurant_eval_df["lexicon_correct"] = ""
restaurant_eval_df["notes"] = ""

display(
    restaurant_eval_df[
        [
            "singlish",
            "reference",
            "cleaned_prediction",
            "lexicon_prediction",
            "cleaned_correct",
            "lexicon_correct",
            "notes",
        ]
    ]
)

In [ ]:
# Provisional semantic-preservation labels:
# yes = meaning preserved; partial = partly preserved; no = meaning changed/lost

cleaned_labels = [
    "no",      # 0: too many ex
    "yes",     # 1: affordable/cheap and good preserved
    "yes",     # 2: very expensive
    "yes",     # 3: within $15
    "partial",# 4: meaning understandable, awkward phrasing
    "yes",     # 5: restaurants nearby
    "yes",     # 6: chicken rice
    "partial",# 7: supper changed to dinner
    "no",      # 8: makan changed to coffee
    "partial",# 9: tabao not normalized
    "yes",     # 10: no seafood
    "yes",     # 11: vegetarian options
    "yes",     # 12: no seafood
    "yes",     # 13: without meat
    "partial",# 14: where can I eat vs where can we eat
    "no",      # 15: ulu changed to empty
    "partial",# 16: question intent changed
    "yes",     # 17: closer location preserved
    "yes",     # 18: travel distance
    "yes",     # 19: walking distance
    "no",      # 20: siew dai lost
    "yes",     # 21: crowd preference
    "yes",     # 22: waiting preference
    "yes",     # 23: invite friends
    "yes",     # 24: shiok = good/nice
]

lexicon_labels = [
    "partial", # 0: expensive preserved, grammar incomplete
    "partial", # 1: "or not" remains
    "yes",     # 2: very expensive
    "yes",     # 3: within $15
    "partial", # 4: awkward phrasing
    "partial", # 5: eat nearby preserved, question phrasing imperfect
    "yes",     # 6: chicken rice
    "yes",     # 7: supper preserved
    "yes",     # 8: eating and queue preference preserved
    "partial", # 9: takeaway intent preserved, awkward phrasing
    "yes",     # 10: no seafood
    "partial", # 11: vegetarian intent preserved, informal wording
    "yes",     # 12: no seafood
    "partial", # 13: meaning preserved, informal wording
    "partial", # 14: subject/group meaning not fully preserved
    "partial", # 15: distance meaning preserved, sentence awkward
    "yes",     # 16: place to eat near SMU
    "yes",     # 17: closer location preserved
    "yes",     # 18: travel distance
    "yes",     # 19: walking distance
    "yes",     # 20: less sugar
    "yes",     # 21: crowd preference
    "yes",     # 22: waiting preference
    "partial", # 23: invite intent preserved, awkward wording
    "partial", # 24: shiok remains untranslated in lexicon input
]

restaurant_eval_df["cleaned_correct"] = cleaned_labels
restaurant_eval_df["lexicon_correct"] = lexicon_labels

score_map = {"yes": 1, "partial": 0.5, "no": 0}

for col in ["cleaned_correct", "lexicon_correct"]:
    restaurant_eval_df[col + "_score"] = (
        restaurant_eval_df[col].map(score_map)
    )

display(
    restaurant_eval_df[
        ["category", "cleaned_correct_score", "lexicon_correct_score"]
    ].groupby("category").mean().round(2)
)

print("Cleaned model mean score:",
      round(restaurant_eval_df["cleaned_correct_score"].mean(), 3))
print("Lexicon pipeline mean score:",
      round(restaurant_eval_df["lexicon_correct_score"].mean(), 3))

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict

# Load your existing data
train_df = pd.read_csv("cs425_data/train_cleaned_pilot.csv")
val_df = pd.read_csv("cs425_data/validation.csv")
test_df = pd.read_csv("cs425_data/test.csv")

# Label convention:
# 1 = Singlish input
# 0 = English translation

def make_style_dataset(df):
    singlish_rows = pd.DataFrame({
        "text": df["singlish"].astype(str),
        "label": 1
    })

    english_rows = pd.DataFrame({
        "text": df["english"].astype(str),
        "label": 0
    })

    combined = pd.concat(
        [singlish_rows, english_rows],
        ignore_index=True
    )

    return Dataset.from_pandas(combined, preserve_index=False)

style_data = DatasetDict({
    "train": make_style_dataset(train_df),
    "validation": make_style_dataset(val_df),
    "test": make_style_dataset(test_df)
})

print(style_data)
print(style_data["train"][0])

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

model_name = "zanelim/singbert"

style_tokenizer = AutoTokenizer.from_pretrained(model_name)

style_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "English", 1: "Singlish"},
    label2id={"English": 0, "Singlish": 1}
)

print("Tokenizer and classifier loaded.")
print("Labels:", style_model.config.id2label)

In [ ]:

from transformers import DataCollatorWithPadding

def tokenize_function(batch):
    return style_tokenizer(
        batch["text"],
        truncation=True,
        max_length=128
    )

tokenized_style_data = style_data.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

data_collator = DataCollatorWithPadding(
    tokenizer=style_tokenizer
)

print(tokenized_style_data)
print(tokenized_style_data["train"][0])


In [ ]:

import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        pos_label=1,
        zero_division=0
    )

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision_singlish": precision,
        "recall_singlish": recall,
        "f1_singlish": f1
    }

training_args = TrainingArguments(
    output_dir="cs425_data/singbert_style_classifier/checkpoints",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_singlish",
    greater_is_better=True,
    logging_steps=20,
    report_to="none",
    seed=42,
    fp16=False
)

trainer = Trainer(
    model=style_model,
    args=training_args,
    train_dataset=tokenized_style_data["train"],
    eval_dataset=tokenized_style_data["validation"],
    processing_class=style_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

train_result = trainer.train()
print(train_result)


In [ ]:

from sklearn.metrics import classification_report, confusion_matrix

test_results = trainer.evaluate(
    tokenized_style_data["test"],
    metric_key_prefix="test"
)

print("Test metrics:")
for key, value in test_results.items():
    if key.startswith("test_"):
        print(f"{key}: {value:.4f}")

test_predictions = trainer.predict(tokenized_style_data["test"])
y_true = test_predictions.label_ids
y_pred = test_predictions.predictions.argmax(axis=-1)

print("\nClassification report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["English", "Singlish"],
        digits=4,
        zero_division=0
    )
)

print("Confusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))


In [ ]:

import os

save_path = "cs425_data/singbert_style_classifier/final_model"
os.makedirs(save_path, exist_ok=True)

trainer.save_model(save_path)
style_tokenizer.save_pretrained(save_path)

print("Model and tokenizer saved to:", save_path)


In [ ]:

import torch

device = next(style_model.parameters()).device
style_model.eval()

def classify_style(text):
    inputs = style_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        logits = style_model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1)[0]

    predicted_id = int(probabilities.argmax().item())

    return {
        "text": text,
        "predicted_style": style_model.config.id2label[predicted_id],
        "english_probability": round(probabilities[0].item(), 4),
        "singlish_probability": round(probabilities[1].item(), 4)
    }

print(classify_style("Eh, this makan place damn ex lah"))
print(classify_style("This restaurant is very expensive"))


In [22]:
import os

for path in [
    "cs425_data/flan_t5_cleaned/final_model",
    "cs425_data/singbert_style_classifier/final_model"
]:
    print(f"\n{path}")
    print("Exists:", os.path.isdir(path))
    if os.path.isdir(path):
        print("Files:", os.listdir(path))


cs425_data/flan_t5_cleaned/final_model
Exists: True
Files: ['training_args.bin', 'tokenizer.json', 'config.json', 'generation_config.json', 'tokenizer_config.json', 'model.safetensors']

cs425_data/singbert_style_classifier/final_model
Exists: True
Files: ['training_args.bin', 'tokenizer.json', 'config.json', 'tokenizer_config.json', 'model.safetensors']


In [23]:

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification
)

# Reload the saved FLAN-T5 normalizer
flan_path = "cs425_data/flan_t5_cleaned/final_model"

flan_tokenizer_saved = AutoTokenizer.from_pretrained(flan_path)
flan_model_saved = AutoModelForSeq2SeqLM.from_pretrained(flan_path)

# Reload the saved SingBERT classifier
singbert_path = "cs425_data/singbert_style_classifier/final_model"

singbert_tokenizer_saved = AutoTokenizer.from_pretrained(singbert_path)
singbert_model_saved = AutoModelForSequenceClassification.from_pretrained(
    singbert_path
)

print("FLAN-T5 reload successful:", type(flan_model_saved).__name__)
print("SingBERT reload successful:", type(singbert_model_saved).__name__)
print("SingBERT labels:", singbert_model_saved.config.id2label)


Loading weights: 100%|████████████████████████████████████████████████████████████| 282/282 [00:00<00:00, 10955.24it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
Loading weights: 100%|████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 11516.83it/s]

FLAN-T5 reload successful: T5ForConditionalGeneration
SingBERT reload successful: BertForSequenceClassification
SingBERT labels: {0: 'English', 1: 'Singlish'}


In [24]:

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
flan_model_saved = flan_model_saved.to(device)
flan_model_saved.eval()

def normalize_singlish(text):
    prompt = "Normalize this Singlish into natural English: " + str(text)

    inputs = flan_tokenizer_saved(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        output_ids = flan_model_saved.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=4
        )

    return flan_tokenizer_saved.decode(
        output_ids[0],
        skip_special_tokens=True
    )

test_queries = [
    "Got any cheap makan places near SMU?",
    "I don't eat seafood, can recommend somewhere with chicken?",
    "Somewhere not too crowded and not damn ex lah.",
    "Can tabao or not?"
]

for query in test_queries:
    print("Input: ", query)
    print("Output:", normalize_singlish(query))
    print("-" * 60)


Input:  Got any cheap makan places near SMU?
Output: Do you have any cheap restaurants near SMU?
------------------------------------------------------------
Input:  I don't eat seafood, can recommend somewhere with chicken?
Output: I don't eat seafood, can you recommend a place with chicken?
------------------------------------------------------------
Input:  Somewhere not too crowded and not damn ex lah.
Output: Somewhere not too crowded and not damn ex.
------------------------------------------------------------
Input:  Can tabao or not?
Output: Can you buy tabao?
------------------------------------------------------------


In [25]:

import re

SINGLISH_RULES = [
    (r"\bdamn\s+ex\b", "very expensive"),
    (r"\btoo\s+ex\b", "too expensive"),
    (r"\bsiew\s+dai\b", "with less sugar"),
    (r"\btabao\b", "get takeaway"),
    (r"\bmakan\s+place\b", "place to eat"),
    (r"\bmakan\b", "eat"),
]

def preprocess_singlish(text):
    result = text
    for pattern, replacement in SINGLISH_RULES:
        result = re.sub(pattern, replacement, result, flags=re.IGNORECASE)
    return result

for query in test_queries:
    preprocessed = preprocess_singlish(query)
    output = normalize_singlish(preprocessed)

    print("Original:     ", query)
    print("Preprocessed: ", preprocessed)
    print("Normalized:   ", output)
    print("-" * 60)


Original:      Got any cheap makan places near SMU?
Preprocessed:  Got any cheap eat places near SMU?
Normalized:    Do you have any cheap food places near SMU?
------------------------------------------------------------
Original:      I don't eat seafood, can recommend somewhere with chicken?
Preprocessed:  I don't eat seafood, can recommend somewhere with chicken?
Normalized:    I don't eat seafood, can you recommend a place with chicken?
------------------------------------------------------------
Original:      Somewhere not too crowded and not damn ex lah.
Preprocessed:  Somewhere not too crowded and not very expensive lah.
Normalized:    Somewhere not too crowded and not very expensive.
------------------------------------------------------------
Original:      Can tabao or not?
Preprocessed:  Can get takeaway or not?
Normalized:    Can you get takeaway?
------------------------------------------------------------


In [26]:

import re

def preprocess_singlish(text):
    result = text.strip()

    # Handle the common takeaway availability question first.
    if re.fullmatch(
        r"can\s+tabao\s+or\s+not[?!.\s]*",
        result,
        flags=re.IGNORECASE
    ):
        return "Is takeaway available?"

    rules = [
        
        (
            r"can\s+lah[,.]?\s+i\s+go\s+there\s+now[.!?\s]*",
            "Okay, I'll go there now."
        ),

        (r"\bsiew\s+dai\b", "with less sugar"),
        (r"\bdamn\s+ex\b", "very expensive"),
        (r"\btoo\s+ex\b", "too expensive"),
        (r"\bmakan\s+place\b", "restaurant"),
        (r"\btabao\b", "takeaway"),
    ]

    for pattern, replacement in rules:
        result = re.sub(
            pattern, replacement, result, flags=re.IGNORECASE
        )

    return result

for query in test_queries:
    preprocessed = preprocess_singlish(query)
    output = normalize_singlish(preprocessed)

    print("Original:    ", query)
    print("Preprocessed:", preprocessed)
    print("Normalized:  ", output)
    print("-" * 60)


Original:     Got any cheap makan places near SMU?
Preprocessed: Got any cheap makan places near SMU?
Normalized:   Do you have any cheap restaurants near SMU?
------------------------------------------------------------
Original:     I don't eat seafood, can recommend somewhere with chicken?
Preprocessed: I don't eat seafood, can recommend somewhere with chicken?
Normalized:   I don't eat seafood, can you recommend a place with chicken?
------------------------------------------------------------
Original:     Somewhere not too crowded and not damn ex lah.
Preprocessed: Somewhere not too crowded and not very expensive lah.
Normalized:   Somewhere not too crowded and not very expensive.
------------------------------------------------------------
Original:     Can tabao or not?
Preprocessed: Is takeaway available?
Normalized:   Is takeaway available?
------------------------------------------------------------


In [27]:

import pandas as pd

diagnostic_cases = [
    # Budget and price
    ("budget", "Got any cheap makan places near SMU?", "cheap; near SMU"),
    ("budget", "This place damn ex lah.", "expensive"),
    ("budget", "Any place not too ex for dinner?", "not expensive"),
    ("budget", "Can find somewhere affordable anot?", "affordable"),
    ("budget", "I want cheap food, not atas restaurants.", "cheap; not fancy"),

    # Location
    ("location", "Any good food near SMU?", "near SMU"),
    ("location", "That place too ulu for me.", "location too remote"),
    ("location", "Can recommend somewhere near the MRT?", "near MRT"),
    ("location", "I don't want to travel so far.", "short travel distance"),

    # Dietary restrictions
    ("dietary", "I don't eat seafood, can recommend somewhere with chicken?", "no seafood; chicken"),
    ("dietary", "Got vegetarian options or not?", "vegetarian options"),
    ("dietary", "No beef please, chicken is fine.", "no beef; chicken allowed"),
    ("dietary", "I cannot eat peanuts.", "avoid peanuts"),

    # Food preferences
    ("food", "Feel like eating something spicy leh.", "spicy food"),
    ("food", "I want makan with lots of vegetables.", "vegetable-rich food"),
    ("food", "Got nice chicken rice around here?", "chicken rice"),
    ("food", "I'm craving something sweet.", "sweet food"),

    # Crowds and queues
    ("crowds", "Somewhere not too crowded and not damn ex lah.", "not crowded; not expensive"),
    ("crowds", "Don't want a noisy place.", "quiet place"),
    ("queues", "The queue damn long, got other options?", "avoid long queue; alternatives"),
    ("queues", "Any place where we don't need to wait so long?", "short waiting time"),

    # Takeaway and dining intent
    ("takeaway", "Can tabao or not?", "takeaway availability"),
    ("takeaway", "I want to dabao dinner back home.", "takeaway dinner"),
    ("dining", "Can eat there or takeaway only?", "dine-in versus takeaway"),
    ("dining", "Can book a table for tonight?", "table reservation tonight"),

    # Social expressions and mixed preferences
    ("social", "Can jio my friends to makan there?", "invite friends to eat there"),
    ("social", "Where can we chill and eat?", "relaxed place to eat"),
    ("preference", "I feel like having chicken but not fried food.", "chicken; not fried"),
    ("preference", "Somewhere casual, not too atas.", "casual; not fancy"),
    ("preference", "I want something filling but not too oily.", "filling; not too oily"),
    ("mixed", "Got good food nearby that's cheap and not crowded?", "good food; nearby; cheap; not crowded"),
]

diagnostic_df = pd.DataFrame(
    diagnostic_cases,
    columns=["category", "input", "expected_meaning"]
)

print("Number of diagnostic queries:", len(diagnostic_df))
display(diagnostic_df)


Number of diagnostic queries: 31


,category,input,expected_meaning
0,budget,Got any cheap makan places near SMU?,cheap; near SMU
1,budget,This place damn ex lah.,expensive
2,budget,Any place not too ex for dinner?,not expensive
3,budget,Can find somewhere affordable anot?,affordable
4,budget,"I want cheap food, not atas restaurants.",cheap; not fancy
5,location,Any good food near SMU?,near SMU
6,location,That place too ulu for me.,location too remote
7,location,Can recommend somewhere near the MRT?,near MRT
8,location,I don't want to travel so far.,short travel distance
9,dietary,"I don't eat seafood, can recommend somewhere w...",no seafood; chicken


In [28]:

from tqdm.auto import tqdm

baseline_outputs = []
hybrid_preprocessed = []
hybrid_outputs = []

for query in tqdm(diagnostic_df["input"], desc="Evaluating queries"):
    # Baseline model
    baseline_outputs.append(normalize_singlish(query))

    # Lexicon-assisted pipeline
    preprocessed = preprocess_singlish(query)
    hybrid_preprocessed.append(preprocessed)

    # Avoid unnecessary generation if preprocessing already
    # produced a complete normalized sentence.
    if preprocessed != query and preprocessed in [
        "Is takeaway available?"
    ]:
        hybrid_outputs.append(preprocessed)
    else:
        hybrid_outputs.append(normalize_singlish(preprocessed))

diagnostic_df["baseline_output"] = baseline_outputs
diagnostic_df["hybrid_preprocessed"] = hybrid_preprocessed
diagnostic_df["hybrid_output"] = hybrid_outputs

diagnostic_df["baseline_preserves_meaning"] = ""
diagnostic_df["hybrid_preserves_meaning"] = ""
diagnostic_df["notes"] = ""

diagnostic_path = "cs425_data/normalizer_diagnostic_results.csv"
diagnostic_df.to_csv(diagnostic_path, index=False)

print("Saved results to:", diagnostic_path)
display(
    diagnostic_df[
        [
            "category",
            "input",
            "expected_meaning",
            "baseline_output",
            "hybrid_output"
        ]
    ]
)


Evaluating queries:   6%|████                                                           | 2/31 [00:20<04:54, 10.14s/it]


KeyboardInterrupt: 

In [ ]:

import re

def preprocess_singlish(text):
    result = text.strip()

    # Exact phrases with clear intended meanings
    phrase_rules = [
        (
            r"can\s+tabao\s+or\s+not[?!.\s]*",
            "Is takeaway available?"
        ),
        (
            r"i\s+want\s+to\s+dabao\s+dinner\s+back\s+home[.!?\s]*",
            "I want to get takeaway for dinner."
        ),
        (
            r"can\s+jio\s+my\s+friends\s+to\s+makan\s+there[.!?\s]*",
            "Can I invite my friends to eat there?"
        ),
    ]

    for pattern, replacement in phrase_rules:
        if re.fullmatch(pattern, result, flags=re.IGNORECASE):
            return replacement

    # Targeted Singlish expressions
    rules = [
        (r"\bsiew\s+dai\b", "with less sugar"),
        (r"\bdamn\s+ex\b", "very expensive"),
        (r"\btoo\s+ex\b", "too expensive"),
        (r"\btoo\s+ulu\b", "too far away"),
        (r"\bulu\b", "remote"),
        (r"\batas\b", "fancy"),
        (r"\bmakan\s+place\b", "restaurant"),
        (r"\btabao\b", "takeaway"),
        (r"\bdabao\b", "takeaway"),
    ]

    for pattern, replacement in rules:
        result = re.sub(
            pattern,
            replacement,
            result,
            flags=re.IGNORECASE
        )

    return result

# Check the four target cases before running the full evaluation
for query in [
    "That place too ulu for me.",
    "I want to dabao dinner back home.",
    "Can jio my friends to makan there?",
    "Somewhere casual, not too atas."
]:
    print("Input: ", query)
    print("Preprocessed:", preprocess_singlish(query))
    print("-" * 50)


In [ ]:

from tqdm.auto import tqdm
import re

complete_phrase_map = {
    "can tabao or not": "Is takeaway available?",
    "i want to dabao dinner back home":
        "I want to get takeaway for dinner.",
    "can jio my friends to makan there":
        "Can I invite my friends to eat there?",
}

def hybrid_normalize(query):
    key = re.sub(r"[.!?]+$", "", query.strip()).casefold()

    if key in complete_phrase_map:
        return complete_phrase_map[key]

    return normalize_singlish(preprocess_singlish(query))


baseline_outputs = []
hybrid_preprocessed = []
hybrid_outputs = []

for query in tqdm(diagnostic_df["input"], desc="Evaluating queries"):
    baseline_outputs.append(normalize_singlish(query))
    hybrid_preprocessed.append(preprocess_singlish(query))
    hybrid_outputs.append(hybrid_normalize(query))

diagnostic_df["baseline_output"] = baseline_outputs
diagnostic_df["hybrid_preprocessed"] = hybrid_preprocessed
diagnostic_df["hybrid_output"] = hybrid_outputs

diagnostic_path = "cs425_data/normalizer_diagnostic_results.csv"
diagnostic_df.to_csv(diagnostic_path, index=False)

display(
    diagnostic_df[
        [
            "category",
            "input",
            "expected_meaning",
            "baseline_output",
            "hybrid_preprocessed",
            "hybrid_output",
        ]
    ]
)
print("Saved results to:", diagnostic_path)


In [ ]:

display(
    diagnostic_df.loc[
        [6, 22, 25, 28],
        [
            "input",
            "expected_meaning",
            "baseline_output",
            "hybrid_preprocessed",
            "hybrid_output",
        ],
    ]
)


In [ ]:

review_df = diagnostic_df[
    ["category", "input", "expected_meaning",
     "baseline_output", "hybrid_output"]
].copy()

review_df["baseline_preserves_meaning"] = ""
review_df["hybrid_preserves_meaning"] = ""
review_df["notes"] = ""

display(review_df)


In [ ]:

# Manually reviewed labels for the latest 31-query results.
# True means all important intended meanings are preserved.

baseline_labels = [
    True, False, True, True, False,
    True, False, True, True, True,
    True, True, True, True, True,
    True, True, True, True, True,
    True, False, False, True, True,
    False, True, True, True, True, True
]

hybrid_labels = [
    True, True, True, True, True,
    True, True, True, True, True,
    True, True, True, True, True,
    True, True, True, True, True,
    True, True, True, True, True,
    True, True, True, True, True, True
]

notes = [
    "",
    "Baseline mistranslates 'damn ex' as disgusting.",
    "",
    "",
    "Baseline changes 'not fancy' to 'not expensive'.",
    "",
    "Baseline misinterprets 'ulu' as expensive.",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "Hybrid correctly translates 'damn ex'.",
    "",
    "",
    "",
    "",
    "Baseline loses takeaway intent.",
    "Baseline loses takeaway intent.",
    "",
    "",
    "Baseline changes eating with friends to meeting.",
    "",
    "",
    "Baseline changes 'atas' to 'shabby'.",
    "",
    ""
]

review_df = diagnostic_df[
    [
        "category",
        "input",
        "expected_meaning",
        "baseline_output",
        "hybrid_output"
    ]
].copy()

review_df["baseline_preserves_meaning"] = baseline_labels
review_df["hybrid_preserves_meaning"] = hybrid_labels
review_df["notes"] = notes

display(review_df)

# Calculate preservation rates
baseline_correct = sum(baseline_labels)
hybrid_correct = sum(hybrid_labels)
total = len(review_df)

print(
    f"Baseline: {baseline_correct}/{total} "
    f"({baseline_correct / total:.1%})"
)
print(
    f"Hybrid: {hybrid_correct}/{total} "
    f"({hybrid_correct / total:.1%})"
)
print(
    "Difference: "
    f"{(hybrid_correct - baseline_correct) / total * 100:+.1f} "
    "percentage points"
)

# Save the reviewed results
review_path = "cs425_data/normalizer_diagnostic_reviewed.csv"
review_df.to_csv(review_path, index=False)
print("Saved reviewed results to:", review_path)


In [ ]:
#new evaluation set :>
import pandas as pd

unseen_queries = [
    ("budget", "This makan place a bit too ex for my budget.",
     "expensive relative to budget"),
    ("budget", "Can find a wallet-friendly lunch spot?",
     "affordable lunch"),
    ("budget", "Don't need atas food, simple and good can already.",
     "simple food; not upscale"),
    ("budget", "The price quite steep leh.",
     "relatively expensive"),
    ("location", "Got anywhere nearby that I can walk to?",
     "nearby; walkable"),
    ("location", "This place damn far from campus.",
     "far from campus"),
    ("location", "Prefer somewhere central, not one corner of Singapore.",
     "central location; not remote"),
    ("location", "Is there a makan place within walking distance?",
     "restaurant within walking distance"),
    ("dietary", "Please recommend somewhere with halal options.",
     "halal options"),
    ("dietary", "I'm vegetarian, so no meat for me.",
     "vegetarian; no meat"),
    ("dietary", "Any dishes without dairy?",
     "dairy-free dishes"),
    ("dietary", "Cannot take spicy food.",
     "avoid spicy food"),
    ("food", "Craving noodles with soup today.",
     "noodles with soup"),
    ("food", "Anything crispy but not deep-fried?",
     "crispy food; not deep-fried"),
    ("food", "I feel like having something soupy and warm.",
     "warm, soupy food"),
    ("food", "Got good nasi lemak nearby?",
     "nasi lemak nearby"),
    ("crowds", "Prefer somewhere peaceful where we can talk.",
     "quiet place suitable for conversation"),
    ("queues", "Don't want to spend half an hour queuing.",
     "avoid long queue"),
    ("takeaway", "Can pack the leftovers for me?",
     "request to pack leftovers"),
    ("mixed", "Need somewhere near campus with vegetarian food that's affordable.",
     "near campus; vegetarian; affordable"),
]

unseen_df = pd.DataFrame(
    unseen_queries,
    columns=["category", "input", "expected_meaning"]
)

unseen_path = "cs425_data/normalizer_unseen_eval.csv"
unseen_df.to_csv(unseen_path, index=False)

print(f"Created {len(unseen_df)} new evaluation queries.")
print("Saved to:", unseen_path)
display(unseen_df)


In [ ]:

from tqdm.auto import tqdm

unseen_df["baseline_output"] = [
    normalize_singlish(q)
    for q in tqdm(unseen_df["input"], desc="Baseline")
]

unseen_df["hybrid_preprocessed"] = [
    preprocess_singlish(q)
    for q in unseen_df["input"]
]

unseen_df["hybrid_output"] = [
    hybrid_normalize(q)
    for q in tqdm(unseen_df["input"], desc="Hybrid")
]

unseen_results_path = "cs425_data/normalizer_unseen_eval_results.csv"
unseen_df.to_csv(unseen_results_path, index=False)

display(
    unseen_df[
        ["category", "input", "expected_meaning",
         "baseline_output", "hybrid_output"]
    ]
)
print("Saved to:", unseen_results_path)


In [ ]:

# Labels for the 20 unseen queries.
# Strict criterion: all important intended meanings must be preserved.

unseen_baseline_labels = [
    True,   # 0: expensive relative to budget
    True,   # 1: wallet-friendly lunch
    False,  # 2: "atas" becomes "a lot of food"
    True,   # 3: "steep" preserves expensive meaning
    True,   # 4: nearby and walkable
    True,   # 5: far from campus
    True,   # 6: central, not remote
    False,  # 7: restaurant specificity is lost
    True,   # 8: halal options
    True,   # 9: vegetarian, no meat
    True,   # 10: dairy-free
    True,   # 11: avoid spicy food
    True,   # 12: noodles with soup
    True,   # 13: crispy but not deep-fried
    True,   # 14: warm and soupy
    False,  # 15: nasi lemak request is lost
    True,   # 16: peaceful place to talk
    True,   # 17: avoids a long wait
    True,   # 18: pack leftovers
    True,   # 19: campus proximity, vegetarian, affordable
]

unseen_hybrid_labels = [
    True,   # 0
    True,   # 1
    True,   # 2: "atas" becomes "fancy"
    True,   # 3
    True,   # 4
    True,   # 5
    True,   # 6
    True,   # 7: restaurant and walking distance preserved
    True,   # 8
    True,   # 9
    True,   # 10
    True,   # 11
    True,   # 12
    True,   # 13
    True,   # 14
    False,  # 15: nasi lemak request is still lost
    True,   # 16
    True,   # 17
    True,   # 18
    True,   # 19
]

unseen_df["baseline_preserves_meaning"] = unseen_baseline_labels
unseen_df["hybrid_preserves_meaning"] = unseen_hybrid_labels

unseen_df["notes"] = [
    "",
    "",
    "Baseline mistranslates atas; hybrid preserves the intended meaning.",
    "",
    "",
    "",
    "",
    "Baseline loses restaurant specificity; hybrid retains it.",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "Both outputs omit the specific nasi lemak request.",
    "",
    "",
    "",
    "",
]

display(unseen_df)

baseline_correct = sum(unseen_baseline_labels)
hybrid_correct = sum(unseen_hybrid_labels)
total = len(unseen_df)

print(
    f"Baseline: {baseline_correct}/{total} "
    f"({baseline_correct / total:.1%})"
)
print(
    f"Hybrid: {hybrid_correct}/{total} "
    f"({hybrid_correct / total:.1%})"
)
print(
    f"Difference: "
    f"{(hybrid_correct - baseline_correct) / total * 100:+.1f} "
    "percentage points"
)

unseen_results_path = "cs425_data/normalizer_unseen_eval_reviewed.csv"
unseen_df.to_csv(unseen_results_path, index=False)
print("Saved reviewed results to:", unseen_results_path)


In [ ]:

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

# Load the saved SingBERT classifier
style_path = "cs425_data/singbert_style_classifier/final_model"

style_tokenizer = AutoTokenizer.from_pretrained(style_path)
style_model = AutoModelForSequenceClassification.from_pretrained(
    style_path
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
style_model = style_model.to(device)
style_model.eval()


def classify_style(text):
    """Return predicted style and class probabilities."""
    inputs = style_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        logits = style_model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1)[0]

    predicted_id = int(probabilities.argmax().item())
    predicted_label = style_model.config.id2label[predicted_id]

    return {
        "label": predicted_label,
        "english_probability": float(probabilities[0].item()),
        "singlish_probability": float(probabilities[1].item())
    }


def restaurant_query_pipeline(text):
    """Classify the query and normalize it when predicted as Singlish."""
    result = classify_style(text)

    if result["label"].casefold() == "singlish":
        normalized = hybrid_normalize(text)
    else:
        normalized = text

    return {
        "original": text,
        "predicted_style": result["label"],
        "english_probability": round(
            result["english_probability"], 4
        ),
        "singlish_probability": round(
            result["singlish_probability"], 4
        ),
        "normalized_query": normalized
    }


print("SingBERT classifier loaded.")
print("Device:", device)
print("Label mapping:", style_model.config.id2label)


In [ ]:

test_queries = [
    "Eh, got cheap makan nearby?",
    "This restaurant is very expensive.",
    "Can tabao or not?",
    "I don't eat seafood, can recommend somewhere with chicken?",
    "Got good nasi lemak nearby?",
    "Could you recommend an affordable restaurant near campus?",
    "Somewhere casual, not too atas.",
    "No beef please, chicken is fine."
]

for query in test_queries:
    result = restaurant_query_pipeline(query)

    print("Input:", result["original"])
    print("Style:", result["predicted_style"])
    print(
        "Probabilities — English:",
        result["english_probability"],
        "| Singlish:",
        result["singlish_probability"]
    )
    print("Output:", result["normalized_query"])
    print("-" * 70)


In [ ]:

import re

def preprocess_singlish(text):
    result = text.strip()

    # Exact phrases with known intended meanings
    phrase_rules = [
        
        (
            r"can\s+lah[,.]?\s+i\s+go\s+there\s+now[.!?\s]*",
            "Okay, I'll go there now."
        ),

        (
            r"can\s+tabao\s+or\s+not[?!.\s]*",
            "Is takeaway available?"
        ),
        (
            r"i\s+want\s+to\s+dabao\s+dinner\s+back\s+home[.!?\s]*",
            "I want to get takeaway for dinner."
        ),
        (
            r"can\s+jio\s+my\s+friends\s+to\s+makan\s+there[.!?\s]*",
            "Can I invite my friends to eat there?"
        ),
        (
            r"eh,\s*got\s+cheap\s+makan\s+nearby[?!.\s]*",
            "Are there any cheap places to eat nearby?"
        ),
        (
            r"got\s+good\s+nasi\s+lemak\s+nearby[?!.\s]*",
            "Are there any good nasi lemak places nearby?"
        ),
    ]

    for pattern, replacement in phrase_rules:
        if re.fullmatch(pattern, result, flags=re.IGNORECASE):
            return replacement

    # Targeted lexical normalization
    rules = [
        (r"\bsiew\s+dai\b", "with less sugar"),
        (r"\bdamn\s+ex\b", "very expensive"),
        (r"\btoo\s+ex\b", "too expensive"),
        (r"\btoo\s+ulu\b", "too far away"),
        (r"\bulu\b", "remote"),
        (r"\batas\b", "fancy"),
        (r"\bmakan\s+place\b", "restaurant"),
        (r"\btabao\b", "takeaway"),
        (r"\bdabao\b", "takeaway"),
    ]

    for pattern, replacement in rules:
        result = re.sub(
            pattern, replacement, result,
            flags=re.IGNORECASE
        )

    return result


In [ ]:

def hybrid_normalize(query):
    key = re.sub(r"[.!?]+$", "", query.strip()).casefold()

    complete_phrase_map = {
        "can tabao or not": "Is takeaway available?",
        "i want to dabao dinner back home":
            "I want to get takeaway for dinner.",
        "can jio my friends to makan there":
            "Can I invite my friends to eat there?",
        "eh, got cheap makan nearby":
            "Are there any cheap places to eat nearby?",
        "got good nasi lemak nearby":
            "Are there any good nasi lemak places nearby?",
    }

    if key in complete_phrase_map:
        return complete_phrase_map[key]

    return normalize_singlish(preprocess_singlish(query))


In [ ]:

for query in test_queries:
    result = restaurant_query_pipeline(query)

    print("Input:", result["original"])
    print("Style:", result["predicted_style"])
    print("Output:", result["normalized_query"])
    print("-" * 60)


In [ ]:

import pandas as pd

fresh_test_queries = [
    "Eh, any good chicken rice near SMU?",
    "Can recommend somewhere cheap for lunch?",
    "I cannot eat peanuts, got anything safe?",
    "Where can I get vegetarian food nearby?",
    "This place quite far leh, got anything closer?",
    "Need somewhere not too crowded for dinner.",
    "Can dabao bubble tea?",
    "Looking for halal food under $15.",
    "Got any good prata places open late?",
    "I don't want spicy food, can?",
    "Somewhere atas for a birthday dinner.",
    "Need a restaurant near the MRT station.",
    "Can recommend somewhere with no seafood?",
    "The queue damn long, got another option?",
    "Want to eat mala but not too spicy.",
    "Could you suggest an inexpensive place near campus?",
    "Please recommend a restaurant with vegetarian options.",
    "No pork please, chicken is okay.",
    "Got dessert places nearby or not?",
    "I want takeaway food that is not too expensive.",
]

fresh_results = []

for query in fresh_test_queries:
    style = classify_style(query)
    normalized = (
        hybrid_normalize(query)
        if style["label"].casefold() == "singlish"
        else query
    )

    fresh_results.append({
        "original_query": query,
        "predicted_style": style["label"],
        "normalized_query": normalized,
    })

fresh_test_df = pd.DataFrame(fresh_results)
display(fresh_test_df)

fresh_test_df.to_csv(
    "cs425_data/normalizer_fresh_test_predictions.csv",
    index=False
)


In [ ]:

import re

PROTECTED_TERMS = [
    "chicken rice",
    "nasi lemak",
    "nasi padang",
    "mee goreng",
    "bubble tea",
    "prata",
    "laksa",
    "mala",
    "satay",
    "seafood",
    "peanuts",
    "vegetarian",
    "halal",
    "no pork",
    "no beef",
    "chicken",
    "beef",
    "pork",
    "under $15",
]

def extract_protected_terms(text):
    found = []
    for term in PROTECTED_TERMS:
        pattern = r"(?<!\w)" + re.escape(term) + r"(?!\w)"
        if re.search(pattern, text, flags=re.IGNORECASE):
            found.append(term)
    return found

def safe_normalize(query):
    result = restaurant_query_pipeline(query)
    normalized = result["normalized_query"]

    required_terms = extract_protected_terms(query)
    missing_terms = [
        term for term in required_terms
        if not re.search(
            r"(?<!\w)" + re.escape(term) + r"(?!\w)",
            normalized,
            flags=re.IGNORECASE
        )
    ]

    if missing_terms:
        return {
            **result,
            "normalized_query": query,
            "fallback_used": True,
            "missing_terms": missing_terms,
        }

    return {
        **result,
        "fallback_used": False,
        "missing_terms": [],
    }


In [ ]:
for query in [
    "Got any good prata places open late?",
    "Want to eat mala but not too spicy.",
    "I cannot eat peanuts, got anything safe?",
]:
    print(safe_normalize(query))
    print("-" * 60)

In [ ]:
guarded_results = [
    safe_normalize(query)
    for query in fresh_test_queries
]

guarded_df = pd.DataFrame(guarded_results)

print("Total queries:", len(guarded_df))
print("Fallbacks:", guarded_df["fallback_used"].sum())
print(
    "Fallback rate:",
    f'{guarded_df["fallback_used"].mean() * 100:.1f}%'
)

display(
    guarded_df[
        ["original", "normalized_query",
         "fallback_used", "missing_terms"]
    ]
)


In [ ]:

def rules_only_normalize(query):
    return preprocess_singlish(query)

rules_comparison = []

for query in fresh_test_queries:
    current = safe_normalize(query)
    rules_only = rules_only_normalize(query)

    rules_comparison.append({
        "original": query,
        "rules_only": rules_only,
        "hybrid_guarded": current["normalized_query"],
        "fallback_used": current["fallback_used"],
    })

rules_comparison_df = pd.DataFrame(rules_comparison)
display(rules_comparison_df)

rules_comparison_df.to_csv(
    "cs425_data/rules_vs_hybrid_comparison.csv",
    index=False
)


In [ ]:

import pandas as pd

# Reuse the comparison dataframe you already created
eval_df = rules_comparison_df.copy()

# Add manual evaluation columns
eval_df["rules_preserves_meaning"] = ""
eval_df["hybrid_preserves_meaning"] = ""
eval_df["notes"] = ""

# Save an editable CSV for annotation
eval_path = "cs425_data/normalizer_manual_evaluation.csv"
eval_df.to_csv(eval_path, index=False)

display(eval_df[
    ["original", "rules_only", "hybrid_guarded",
     "rules_preserves_meaning", "hybrid_preserves_meaning", "notes"]
])

print("Saved to:", eval_path)


In [ ]:

import pandas as pd

eval_path = "cs425_data/normalizer_manual_evaluation.csv"
eval_df = pd.read_csv(eval_path)

# Rules-only preserves the original meaning in all 20 examples.
rules_labels = ["yes"] * 20

# Hybrid preserves meaning except for the two questionable outputs.
hybrid_labels = [
    "yes",  # 0: "Hey" does not change the request
    "yes",  # 1: unchanged
    "yes",  # 2: peanut restriction preserved
    "yes",  # 3: unchanged
    "yes",  # 4: distance and closer alternative preserved
    "yes",  # 5: unchanged
    "yes",  # 6: takeaway intent preserved
    "yes",  # 7: halal and budget preserved
    "yes",  # 8: original restored by fallback
    "no",   # 9: "can I?" is awkward and incomplete
    "yes",  # 10: atas -> fancy
    "yes",  # 11: unchanged
    "yes",  # 12: seafood exclusion preserved
    "yes",  # 13: long queue and alternative preserved
    "yes",  # 14: mala and spice constraint preserved
    "yes",  # 15: unchanged
    "yes",  # 16: unchanged
    "yes",  # 17: pork exclusion and chicken preference preserved
    "no",   # 18: availability question becomes "Have you found..."
    "yes",  # 19: takeaway and price intent preserved
]

notes = [
    "Meaning preserved; minor conversational filler added.",
    "Meaning preserved; unchanged.",
    "Peanut restriction and request for a safe option preserved.",
    "Meaning preserved; unchanged.",
    "Distance and request for a closer option preserved.",
    "Meaning preserved; unchanged.",
    "Takeaway intent and bubble tea preserved.",
    "Halal requirement and $15 budget preserved.",
    "Prata and late opening intent preserved by fallback.",
    "Output is awkward and does not clearly complete the intended question.",
    "Preference for a fancy birthday dinner preserved.",
    "Meaning preserved; unchanged.",
    "Seafood exclusion preserved.",
    "Long queue and request for an alternative preserved.",
    "Mala and the preference for lower spice preserved.",
    "Meaning preserved; unchanged.",
    "Vegetarian options preserved.",
    "Pork exclusion and chicken preference preserved.",
    "Changes a request for nearby dessert places into a question about finding them.",
    "Takeaway and affordability preserved.",
]

assert len(eval_df) == len(rules_labels) == len(hybrid_labels) == len(notes) == 20

eval_df["rules_preserves_meaning"] = rules_labels
eval_df["hybrid_preserves_meaning"] = hybrid_labels
eval_df["notes"] = notes

eval_df.to_csv(eval_path, index=False)

display(eval_df[
    ["original", "rules_preserves_meaning",
     "hybrid_preserves_meaning", "notes"]
])

print("Saved completed evaluation to:", eval_path)


In [ ]:

summary = pd.DataFrame({
    "System": ["Rules-only", "Hybrid + guard"],
    "Meaning preserved": [
        (eval_df["rules_preserves_meaning"] == "yes").sum(),
        (eval_df["hybrid_preserves_meaning"] == "yes").sum(),
    ],
})

summary["Total"] = len(eval_df)
summary["Preservation rate (%)"] = (
    summary["Meaning preserved"] / summary["Total"] * 100
).round(1)

display(summary)


In [ ]:

import pandas as pd

final_test_queries = [
    "Any nice chicken rice stall around Bugis?",
    "Got vegetarian makan near the school?",
    "Can help me find food below ten dollars?",
    "Don't recommend anything with prawns please.",
    "Is there somewhere to eat after midnight?",
    "This cafe very crowded sia.",
    "I want fish soup but no milk, can?",
    "Got halal nasi padang near the MRT?",
    "Can find somewhere not so noisy for my parents?",
    "The food there quite pricey leh.",
    "Any place selling wanton mee nearby?",
    "Can pack the leftovers for me?",
    "Need a quiet cafe where I can study.",
    "No eggs please, I have an allergy.",
    "Where to get good kaya toast?",
    "I prefer somewhere not too far from campus.",
    "Can recommend a place with vegetarian laksa?",
    "Got affordable food open on Sunday?",
    "Please find a restaurant that serves chicken and rice.",
    "Want to eat something warm, maybe porridge.",
]

final_test_results = []

for query in final_test_queries:
    style = classify_style(query)

    # Compare both systems without modifying either one.
    rules_output = rules_only_normalize(query)
    hybrid_output = safe_normalize(query)["normalized_query"]

    final_test_results.append({
        "original": query,
        "predicted_style": style["label"],
        "rules_only": rules_output,
        "hybrid_guarded": hybrid_output,
        "rules_preserves_meaning": "",
        "hybrid_preserves_meaning": "",
        "notes": "",
    })

final_test_df = pd.DataFrame(final_test_results)

final_test_path = "cs425_data/normalizer_final_test_predictions.csv"
final_test_df.to_csv(final_test_path, index=False)

display(final_test_df)
print("Saved to:", final_test_path)


In [ ]:

import pandas as pd

final_test_path = "cs425_data/normalizer_final_test_predictions.csv"
final_test_df = pd.read_csv(final_test_path)

rules_labels = ["yes"] * 20

hybrid_labels = [
    "yes", "yes", "yes", "yes", "yes",
    "yes", "no", "yes", "yes", "yes",
    "yes", "yes", "yes", "yes", "yes",
    "yes", "yes", "yes", "yes", "yes",
]

notes = [
    "Chicken rice and Bugis preserved.",
    "Vegetarian food and location preserved.",
    "Budget intent preserved.",
    "Prawn exclusion preserved.",
    "Late-night dining intent preserved.",
    "Crowdedness preserved; filler omitted.",
    "Awkward, incomplete question ending.",
    "Halal, nasi padang and MRT preserved.",
    "Quietness and parents' needs preserved.",
    "Price meaning preserved.",
    "Wanton mee and nearby location preserved.",
    "Packing leftovers preserved.",
    "Unchanged; meaning preserved.",
    "Egg allergy preserved.",
    "Kaya toast preserved.",
    "Distance preference preserved.",
    "Vegetarian laksa preserved.",
    "Affordability and Sunday availability preserved.",
    "Chicken and rice preserved.",
    "Warm food and porridge preference preserved.",
]

assert len(final_test_df) == 20

final_test_df["rules_preserves_meaning"] = rules_labels
final_test_df["hybrid_preserves_meaning"] = hybrid_labels
final_test_df["notes"] = notes

final_test_df.to_csv(final_test_path, index=False)

summary = pd.DataFrame({
    "System": ["Rules-only", "Hybrid + guard"],
    "Meaning preserved": [
        (final_test_df["rules_preserves_meaning"] == "yes").sum(),
        (final_test_df["hybrid_preserves_meaning"] == "yes").sum(),
    ],
})
summary["Total"] = len(final_test_df)
summary["Preservation rate (%)"] = (
    summary["Meaning preserved"] / summary["Total"] * 100
).round(1)

display(summary)
print("Saved reviewed results to:", final_test_path)


In [ ]:

def final_normalize(query):
    """
    Conservative rules-first normalizer.
    Preserves the original query when no known rule applies.
    """
    original = query.strip()
    normalized = preprocess_singlish(original)

    return {
        "original_query": original,
        "normalized_query": normalized,
        "rule_applied": normalized != original,
        "normalizer": (
            "rules" if normalized != original else "original_preserved"
        ),
    }


# Smoke test
pipeline_test_queries = [
    "Somewhere casual, not too atas.",
    "Can tabao or not?",
    "Got any good prata places open late?",
    "Want to eat mala but not too spicy.",
    "I cannot eat peanuts, got anything safe?",
    "Could you recommend an affordable restaurant near campus?",
]

for query in pipeline_test_queries:
    print(final_normalize(query))
    print("-" * 60)


In [ ]:

import re

def preprocess_singlish(text):
    result = text.strip()

    # Handle common complete expressions first.
    phrase_rules = [
        
        (
            r"can\s+lah[,.]?\s+i\s+go\s+there\s+now[.!?\s]*",
            "Okay, I'll go there now."
        ),

        (
            r"can\s+tabao\s+or\s+not[?!.\s]*",
            "Is takeaway available?"
        ),
        (
            r"i\s+want\s+to\s+dabao\s+dinner\s+back\s+home[.!?\s]*",
            "I want to get takeaway for dinner."
        ),
        (
            r"can\s+jio\s+my\s+friends\s+to\s+makan\s+there[.!?\s]*",
            "Can I invite my friends to eat there?"
        ),
        (
            r"eh,\s*got\s+cheap\s+makan\s+nearby[?!.\s]*",
            "Are there any cheap places to eat nearby?"
        ),
    ]

    for pattern, replacement in phrase_rules:
        if re.fullmatch(pattern, result, flags=re.IGNORECASE):
            return replacement

    # Normalize known Singlish vocabulary.
    lexical_rules = [
        (r"\bsiew\s+dai\b", "with less sugar"),
        (r"\bdamn\s+ex\b", "very expensive"),
        (r"\btoo\s+ex\b", "too expensive"),
        (r"\btoo\s+ulu\b", "too far away"),
        (r"\bulu\b", "remote"),
        (r"\batas\b", "fancy"),
        (r"\bmakan\s+place\b", "restaurant"),
        (r"\bdabao\b", "takeaway"),
        (r"\btabao\b", "takeaway"),
        (r"\bleh\b", ""),
        (r"\blah\b", ""),
        (r"\bsia\b", ""),
    ]

    for pattern, replacement in lexical_rules:
        result = re.sub(
            pattern, replacement, result, flags=re.IGNORECASE
        )

    # Normalize a few common sentence constructions.
    result = re.sub(
        r"^\s*got\s+any\s+good\s+(.+?)\s+places\s+open\s+late\??\s*$",
        r"Are there any good \1 places open late?",
        result,
        flags=re.IGNORECASE,
    )

    result = re.sub(
        r"\bi\s+cannot\s+eat\s+(.+?),\s*got\s+anything\s+safe\??",
        r"I cannot eat \1. Are there any suitable options?",
        result,
        flags=re.IGNORECASE,
    )

    # Clean up spaces left by removing discourse particles.
    result = re.sub(r"\s+([,.!?])", r"\1", result)
    result = re.sub(r"\s{2,}", " ", result).strip()

    return result


In [ ]:

test_queries = [
    "Somewhere casual, not too atas.",
    "Can tabao or not?",
    "Got any good prata places open late?",
    "Want to eat mala but not too spicy.",
    "I cannot eat peanuts, got anything safe?",
    "Eh, got cheap makan nearby?",
    "This place damn ex lah.",
    "Could you recommend an affordable restaurant near campus?",
]

for query in test_queries:
    print("Original:", query)
    print("Result:  ", final_normalize(query)["normalized_query"])
    print("-" * 60)



In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

style_path = "cs425_data/singbert_style_classifier/final_model"

style_tokenizer = AutoTokenizer.from_pretrained(style_path)
style_model = AutoModelForSequenceClassification.from_pretrained(style_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
style_model = style_model.to(device)
style_model.eval()

print("Device:", device)
print("Labels:", style_model.config.id2label)
print("SingBERT loaded successfully.")


In [ ]:

def classify_style(text):
    inputs = style_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        logits = style_model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1)[0]

    predicted_id = int(probabilities.argmax().item())
    label = style_model.config.id2label[predicted_id]

    return {
        "label": label,
        "english_probability": float(probabilities[0].item()),
        "singlish_probability": float(probabilities[1].item())
    }


def integrated_normalize(query):
    original = query.strip()
    style = classify_style(original)

    if style["label"].casefold() == "singlish":
        normalized = preprocess_singlish(original)
        normalizer = (
            "rules" if normalized != original
            else "original_preserved"
        )
    else:
        normalized = original
        normalizer = "original_preserved"

    return {
        "original_query": original,
        "predicted_style": style["label"],
        "english_probability": round(
            style["english_probability"], 4
        ),
        "singlish_probability": round(
            style["singlish_probability"], 4
        ),
        "normalized_query": normalized,
        "rule_applied": normalized != original,
        "normalizer": normalizer
    }

print("Integrated pipeline defined.")


In [ ]:
#smoke test

test_queries = [
    "Somewhere casual, not too atas.",
    "Can tabao or not?",
    "Got any good prata places open late?",
    "Want to eat mala but not too spicy.",
    "I cannot eat peanuts, got anything safe?",
    "Eh, got cheap makan nearby?",
    "This place damn ex lah.",
    "Could you recommend an affordable restaurant near campus?",
    "No beef please, chicken is fine."
]

for query in test_queries:
    result = integrated_normalize(query)

    print("Input:     ", result["original_query"])
    print("Style:     ", result["predicted_style"])
    print("Normalized:", result["normalized_query"])
    print("Normalizer:", result["normalizer"])
    print("-" * 60)


In [ ]:

import re

PROTECTED_TERMS = [
    "chicken rice",
    "nasi lemak",
    "nasi padang",
    "mee goreng",
    "bubble tea",
    "prata",
    "laksa",
    "mala",
    "satay",
    "seafood",
    "peanuts",
    "vegetarian",
    "halal",
    "no pork",
    "no beef",
    "chicken",
    "beef",
    "pork",
    "under $15",
]

def extract_protected_terms(text):
    found = []

    for term in PROTECTED_TERMS:
        pattern = r"(?<!\w)" + re.escape(term) + r"(?!\w)"

        if re.search(pattern, text, flags=re.IGNORECASE):
            found.append(term)

    return found



import re

NEGATION_PATTERNS = [
    r"\bno\b",
    r"\bnot\b",
    r"\bnever\b",
    r"\bdon't\b",
    r"\bdo not\b",
    r"\bdoesn't\b",
    r"\bwithout\b",
    r"\bcan't\b",
    r"\bcannot\b",
    r"\bavoid\b",
    r"\bexcept\b",
]

def meaning_safety_check(original, normalized):
    original_lower = original.casefold()
    normalized_lower = normalized.casefold()

    # Preserve whether the query contains an explicit negation.
    original_negation = any(
        re.search(pattern, original_lower)
        for pattern in NEGATION_PATTERNS
    )
    normalized_negation = any(
        re.search(pattern, normalized_lower)
        for pattern in NEGATION_PATTERNS
    )

    if original_negation and not normalized_negation:
        return False, "negation_lost"

    # Preserve important food, dietary, and budget terms.
    original_terms = extract_protected_terms(original)
    normalized_terms = extract_protected_terms(normalized)

    missing_terms = [
        term for term in original_terms
        if term.casefold() not in normalized_lower
    ]

    if missing_terms:
        return False, "protected_terms_lost: " + ", ".join(missing_terms)

    # Avoid changing a question into a statement, or vice versa.
    original_is_question = original.rstrip().endswith("?")
    normalized_is_question = normalized.rstrip().endswith("?")

    if original_is_question != normalized_is_question:
        return False, "question_type_changed"

    return True, "passed"


def safe_integrated_normalize(query):
    result = integrated_normalize(query)

    original = result["original_query"]
    normalized = result["normalized_query"]

    safe, reason = meaning_safety_check(original, normalized)
    fallback_used = not safe

    if fallback_used:
        result["normalized_query"] = original
        result["normalizer"] = "safety_fallback"
        result["rule_applied"] = False

    result["fallback_used"] = fallback_used
    result["safety_reason"] = reason

    return result

print("Updated safety checks defined.")

In [ ]:
def integrated_normalize(query):
    original = query.strip()
    style = classify_style(original)

    if style["label"].casefold() == "singlish":
        normalized = preprocess_singlish(original)
        normalizer = (
            "rules" if normalized != original
            else "original_preserved"
        )
    else:
        normalized = original
        normalizer = "original_preserved"

    return {
        "original_query": original,
        "predicted_style": style["label"],
        "english_probability": round(
            style["english_probability"], 4
        ),
        "singlish_probability": round(
            style["singlish_probability"], 4
        ),
        "normalized_query": normalized,
        "rule_applied": normalized != original,
        "normalizer": normalizer
    }

print("Integrated pipeline defined.")

In [ ]:

safety_test_queries = [
    "Got any good prata places open late?",
    "Want to eat mala but not too spicy.",
    "I cannot eat peanuts, got anything safe?",
    "I don't eat seafood, can recommend somewhere with chicken?",
    "No beef please, chicken is fine.",
    "Could you recommend an affordable restaurant near campus?",
]

for query in safety_test_queries:
    result = safe_integrated_normalize(query)

    print("Input:       ", result["original_query"])
    print("Style:       ", result["predicted_style"])
    print("Output:      ", result["normalized_query"])
    print("Fallback:    ", result["fallback_used"])
    print("Missing terms:", result["missing_protected_terms"])
    print("-" * 60)


In [ ]:

safety_test_queries = [
    "I don't eat seafood, can recommend somewhere with chicken?",
    "Got vegetarian options or not?",
    "I want chicken rice.",
    "Can tabao or not?",
    "I want food without peanuts."
]

for query in safety_test_queries:
    result = safe_integrated_normalize(query)
    print("Original: ", result["original_query"])
    print("Output:   ", result["normalized_query"])
    print("Fallback: ", result["fallback_used"])
    print("Reason:   ", result["safety_reason"])
    print("-" * 50)

In [ ]:

import pandas as pd
from pathlib import Path

eval_path = Path("cs425_data/normalizer_final_test_predictions.csv")

if eval_path.exists():
    eval_df = pd.read_csv(eval_path)
    print("Rows:", len(eval_df))
    print("Columns:", eval_df.columns.tolist())
    display(eval_df.head(10))
else:
    print("File not found:", eval_path)
    print("Available normalizer CSV files:")
    for path in sorted(Path("cs425_data").glob("*normalizer*.csv")):
        print(path)


In [ ]:

import pandas as pd

eval_path = "cs425_data/normalizer_final_test_predictions.csv"
eval_df = pd.read_csv(eval_path)

def yes_count(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .eq("yes")
        .sum()
    )

summary = pd.DataFrame([
    {
        "System": "Rules-only",
        "Meaning preserved": yes_count(
            eval_df["rules_preserves_meaning"]
        ),
        "Total": len(eval_df),
    },
    {
        "System": "Hybrid + guard",
        "Meaning preserved": yes_count(
            eval_df["hybrid_preserves_meaning"]
        ),
        "Total": len(eval_df),
    },
])

summary["Preservation rate (%)"] = (
    summary["Meaning preserved"] / summary["Total"] * 100
).round(1)

display(summary)


In [ ]:

# How often did each system change the original text?
original = eval_df["original"].fillna("").str.strip()
rules_output = eval_df["rules_only"].fillna("").str.strip()
hybrid_output = eval_df["hybrid_guarded"].fillna("").str.strip()

rules_changed = rules_output.ne(original)
hybrid_changed = hybrid_output.ne(original)

coverage = pd.DataFrame([
    {
        "System": "Rules-only",
        "Queries changed": int(rules_changed.sum()),
        "Total queries": len(eval_df),
        "Coverage (%)": round(rules_changed.mean() * 100, 1),
    },
    {
        "System": "Hybrid + guard",
        "Queries changed": int(hybrid_changed.sum()),
        "Total queries": len(eval_df),
        "Coverage (%)": round(hybrid_changed.mean() * 100, 1),
    },
])

display(coverage)


In [ ]:

hybrid_failures = eval_df[
    eval_df["hybrid_preserves_meaning"]
    .astype(str).str.strip().str.lower().eq("no")
][["original", "hybrid_guarded", "notes"]]

display(hybrid_failures)


In [ ]:

def classify_style(text):
    """Return predicted style and class probabilities."""
    inputs = style_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        logits = style_model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1)[0]

    predicted_id = int(probabilities.argmax().item())
    predicted_label = style_model.config.id2label[predicted_id]

    return {
        "label": predicted_label,
        "english_probability": float(probabilities[0].item()),
        "singlish_probability": float(probabilities[1].item())
    }

print("classify_style defined.")


In [29]:
print("style_tokenizer loaded:", "style_tokenizer" in globals())
print("style_model loaded:", "style_model" in globals())
print("device defined:", "device" in globals())

style_tokenizer loaded: True
style_model loaded: True
device defined: True


In [30]:

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

style_checkpoint = "/common/home/projectgrps/CS425/CS425G8/cs425_data/singbert_style_classifier/final_model"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

style_tokenizer = AutoTokenizer.from_pretrained(style_checkpoint)
style_model = AutoModelForSequenceClassification.from_pretrained(
    style_checkpoint
).to(device)

style_model.eval()

print("SingBERT loaded successfully.")
print("Labels:", style_model.config.id2label)


Using device: cpu


Loading weights: 100%|████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 10924.08it/s]

SingBERT loaded successfully.
Labels: {0: 'English', 1: 'Singlish'}


In [31]:
import re
def preprocess_singlish(text):
    result = text.strip()

    # Handle common complete expressions first.
    phrase_rules = [
        
        (
            r"can\s+lah[,.]?\s+i\s+go\s+there\s+now[.!?\s]*",
            "Okay, I'll go there now."
        ),

        (
            r"can\s+tabao\s+or\s+not[?!.\s]*",
            "Is takeaway available?"
        ),
        (
            r"i\s+want\s+to\s+dabao\s+dinner\s+back\s+home[.!?\s]*",
            "I want to get takeaway for dinner."
        ),
        (
            r"can\s+jio\s+my\s+friends\s+to\s+makan\s+there[.!?\s]*",
            "Can I invite my friends to eat there?"
        ),
        (
            r"eh,\s*got\s+cheap\s+makan\s+nearby[?!.\s]*",
            "Are there any cheap places to eat nearby?"
        ),
    ]

    for pattern, replacement in phrase_rules:
        if re.fullmatch(pattern, result, flags=re.IGNORECASE):
            return replacement

    # Normalize known Singlish vocabulary.
    lexical_rules = [
        (r"\bsiew\s+dai\b", "with less sugar"),
        (r"\bdamn\s+ex\b", "very expensive"),
        (r"\btoo\s+ex\b", "too expensive"),
        (r"\btoo\s+ulu\b", "too far away"),
        (r"\bulu\b", "remote"),
        (r"\batas\b", "fancy"),
        (r"\bmakan\s+place\b", "restaurant"),
        (r"\bdabao\b", "takeaway"),
        (r"\btabao\b", "takeaway"),
        (r"\bleh\b", ""),
        (r"\blah\b", ""),
        (r"\bsia\b", ""),
    ]

    for pattern, replacement in lexical_rules:
        result = re.sub(
            pattern, replacement, result, flags=re.IGNORECASE
        )

    # Normalize a few common sentence constructions.
    result = re.sub(
        r"^\s*got\s+any\s+good\s+(.+?)\s+places\s+open\s+late\??\s*$",
        r"Are there any good \1 places open late?",
        result,
        flags=re.IGNORECASE,
    )

    result = re.sub(
        r"\bi\s+cannot\s+eat\s+(.+?),\s*got\s+anything\s+safe\??",
        r"I cannot eat \1. Are there any suitable options?",
        result,
        flags=re.IGNORECASE,
    )

    # Clean up spaces left by removing discourse particles.
    result = re.sub(r"\s+([,.!?])", r"\1", result)
    result = re.sub(r"\s{2,}", " ", result).strip()

    return result


In [32]:
def integrated_normalize(query):
    original = query.strip()
    style = classify_style(original)

    if style["label"].casefold() == "singlish":
        normalized = preprocess_singlish(original)
        normalizer = (
            "rules" if normalized != original
            else "original_preserved"
        )
    else:
        normalized = original
        normalizer = "original_preserved"

    return {
        "original_query": original,
        "predicted_style": style["label"],
        "english_probability": round(
            style["english_probability"], 4
        ),
        "singlish_probability": round(
            style["singlish_probability"], 4
        ),
        "normalized_query": normalized,
        "rule_applied": normalized != original,
        "normalizer": normalizer
    }

print("Integrated pipeline defined.")

Integrated pipeline defined.


In [33]:
test_queries = [
    "Can tabao or not?",
    "I don't eat seafood.",
    "Those cafes quite ulu I think.",
    "Got vegetarian options or not?",
]

for query in test_queries:
    print(integrated_normalize(query))
    print("-" * 50)

{'original_query': 'Can tabao or not?', 'predicted_style': 'Singlish', 'english_probability': 0.0039, 'singlish_probability': 0.9961, 'normalized_query': 'Is takeaway available?', 'rule_applied': True, 'normalizer': 'rules'}
--------------------------------------------------
{'original_query': "I don't eat seafood.", 'predicted_style': 'English', 'english_probability': 0.9924, 'singlish_probability': 0.0076, 'normalized_query': "I don't eat seafood.", 'rule_applied': False, 'normalizer': 'original_preserved'}
--------------------------------------------------
{'original_query': 'Those cafes quite ulu I think.', 'predicted_style': 'Singlish', 'english_probability': 0.0041, 'singlish_probability': 0.9959, 'normalized_query': 'Those cafes quite remote I think.', 'rule_applied': True, 'normalizer': 'rules'}
--------------------------------------------------
{'original_query': 'Got vegetarian options or not?', 'predicted_style': 'Singlish', 'english_probability': 0.008, 'singlish_probabilit

In [34]:
print("safe_integrated_normalize defined:",
      "safe_integrated_normalize" in globals())

print("normalize_singlish defined:",
      "normalize_singlish" in globals())

print("flan_normalize defined:",
      "flan_normalize" in globals())

safe_integrated_normalize defined: True
normalize_singlish defined: True
flan_normalize defined: True


In [35]:

import re

FOOD_AND_CONSTRAINT_TERMS = [
    "chicken rice", "nasi lemak", "nasi padang", "mee goreng",
    "bubble tea", "prata", "laksa", "mala", "satay",
    "seafood", "peanuts", "vegetarian", "halal",
    "chicken", "beef", "pork", "no chilli", "no chili",
    "no pork", "no beef"
]

def basic_safety_check(original, candidate):
    original_lower = original.casefold()
    candidate_lower = candidate.casefold()

    # Ensure explicit food terms and constraints aren't dropped.
    for term in FOOD_AND_CONSTRAINT_TERMS:
        if re.search(r"\b" + re.escape(term) + r"\b", original_lower):
            if not re.search(r"\b" + re.escape(term) + r"\b", candidate_lower):
                return False

    # Preserve explicit negative constraints.
    negation_patterns = [
        r"\bno\b", r"\bnot\b", r"\bnever\b",
        r"\bdon't\b", r"\bdoesn't\b", r"\bdidn't\b",
        r"\bcannot\b", r"\bcan't\b", r"\bwithout\b"
    ]

    # "or not?" is a Singlish question marker, not a constraint.
    original_for_negation = re.sub(
        r"\bor not\s*\??\s*$", "", original_lower
    )

    original_has_negation = any(
        re.search(pattern, original_for_negation)
        for pattern in negation_patterns
    )
    candidate_has_negation = any(
        re.search(pattern, candidate_lower)
        for pattern in negation_patterns
    )

    if original_has_negation and not candidate_has_negation:
        return False

    # Preserve whether the original is explicitly a question.
    if original.rstrip().endswith("?") != candidate.rstrip().endswith("?"):
        return False

    return True


def safe_integrated_normalize(query):
    result = integrated_normalize(query)
    original = result["original_query"]

    # Prefer existing rules when they successfully change the query.
    if result["rule_applied"]:
        result["fallback_used"] = False
        result["safety_reason"] = "Existing rule applied"
        return result

    # Otherwise, try the fine-tuned FLAN-T5.
    candidate = flan_normalize(original).strip()

    if candidate and basic_safety_check(original, candidate):
        result["normalized_query"] = candidate
        result["normalizer"] = "flan_t5"
        result["rule_applied"] = False
        result["fallback_used"] = False
        result["safety_reason"] = "Basic checks passed"
    else:
        result["normalized_query"] = original
        result["normalizer"] = "safety_fallback"
        result["rule_applied"] = False
        result["fallback_used"] = True
        result["safety_reason"] = "Empty output or basic safety check failed"

    return result

print("Safe normalizer defined.")


Safe normalizer defined.


In [36]:

test_queries = [
    "Can tabao or not?",
    "I don't eat seafood.",
    "Those cafes quite ulu I think.",
    "Got vegetarian options or not?",
    "I want chicken rice, no chilli please.",
    "Can lah, I go there now."
]

for query in test_queries:
    result = safe_integrated_normalize(query)
    print("Input:     ", result["original_query"])
    print("Normalized:", result["normalized_query"])
    print("Method:    ", result["normalizer"])
    print("Fallback:  ", result["fallback_used"])
    print("-" * 60)


Input:      Can tabao or not?
Normalized: Is takeaway available?
Method:     rules
Fallback:   False
------------------------------------------------------------
Input:      I don't eat seafood.
Normalized: I don't eat seafood.
Method:     flan_t5
Fallback:   False
------------------------------------------------------------
Input:      Those cafes quite ulu I think.
Normalized: Those cafes quite remote I think.
Method:     rules
Fallback:   False
------------------------------------------------------------
Input:      Got vegetarian options or not?
Normalized: Have you got vegetarian options?
Method:     flan_t5
Fallback:   False
------------------------------------------------------------
Input:      I want chicken rice, no chilli please.
Normalized: I want chicken rice, no chilli please.
Method:     flan_t5
Fallback:   False
------------------------------------------------------------
Input:      Can lah, I go there now.
Normalized: Okay, I'll go there now.
Method:     rules
Fallbac

In [37]:
print(safe_integrated_normalize("Can lah, I go there now."))

{'original_query': 'Can lah, I go there now.', 'predicted_style': 'Singlish', 'english_probability': 0.0038, 'singlish_probability': 0.9962, 'normalized_query': "Okay, I'll go there now.", 'rule_applied': True, 'normalizer': 'rules', 'fallback_used': False, 'safety_reason': 'Existing rule applied'}


In [38]:
from pathlib import Path

checkpoint = Path(
    "/common/home/projectgrps/CS425/CS425G8/cs425_data/flan_t5_cleaned/final_model"
)

print("Checkpoint exists:", checkpoint.exists())
print("Model weights exist:", (checkpoint / "model.safetensors").exists())

Checkpoint exists: True
Model weights exist: True


In [39]:
import inspect

print(inspect.getsource(safe_integrated_normalize))

def safe_integrated_normalize(query):
    result = integrated_normalize(query)
    original = result["original_query"]

    # Prefer existing rules when they successfully change the query.
    if result["rule_applied"]:
        result["fallback_used"] = False
        result["safety_reason"] = "Existing rule applied"
        return result

    # Otherwise, try the fine-tuned FLAN-T5.
    candidate = flan_normalize(original).strip()

    if candidate and basic_safety_check(original, candidate):
        result["normalized_query"] = candidate
        result["normalizer"] = "flan_t5"
        result["rule_applied"] = False
        result["fallback_used"] = False
        result["safety_reason"] = "Basic checks passed"
    else:
        result["normalized_query"] = original
        result["normalizer"] = "safety_fallback"
        result["rule_applied"] = False
        result["fallback_used"] = True
        result["safety_reason"] = "Empty output or basic safety check failed"

    return result

In [40]:
print(safe_integrated_normalize("Can lah, I go there now."))

{'original_query': 'Can lah, I go there now.', 'predicted_style': 'Singlish', 'english_probability': 0.0038, 'singlish_probability': 0.9962, 'normalized_query': "Okay, I'll go there now.", 'rule_applied': True, 'normalizer': 'rules', 'fallback_used': False, 'safety_reason': 'Existing rule applied'}


In [41]:
print("flan_normalize defined:", "flan_normalize" in globals())

flan_normalize defined: True


In [42]:
print(flan_normalize("Got vegetarian options or not?"))

Have you got vegetarian options?


In [43]:
from pathlib import Path
import json

notebook_path = Path.home() / "varidhi_flant5.ipynb"
nb = json.loads(notebook_path.read_text())

print("Total cells:", len(nb["cells"]))
for i, cell in enumerate(nb["cells"]):
    if cell["cell_type"] == "markdown":
        text = "".join(cell["source"]).strip().replace("\n", " ")
        print(f"{i}: MARKDOWN — {text[:100]}")
    else:
        code = "".join(cell["source"]).strip().split("\n")
        first_line = code[0] if code else "(empty)"
        print(f"{i}: CODE — {first_line[:100]}")

Total cells: 131
0: CODE — import sys
1: CODE — import sys
2: CODE — import torch
3: CODE — import sys
4: CODE — import torch
5: CODE — import torch
6: CODE — from datasets import load_dataset
7: CODE — import os
8: CODE — import pandas as pd
9: CODE — from sklearn.model_selection import train_test_split
10: CODE — train_df.to_csv("cs425_data/train.csv", index=False)
11: CODE — import pandas as pd
12: CODE — from IPython.display import display
13: CODE — # Evaluate the existing predictions without rerunning the model
14: CODE — import re
15: CODE — import re
16: CODE — # Test the fine-tuned checkpoint with the training prompt
17: CODE — print("SINGLISH_RULES defined:", "SINGLISH_RULES" in globals())
18: CODE — import sys
19: CODE — import transformers
20: CODE — import pandas as pd
21: CODE — from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
22: CODE — train_result = trainer.train()
23: CODE — FINAL_MODEL_DIR = "cs425_data/flan_t5_singlish_final"
24: CODE — import torch